In [ ]:
# 1.1 导入库

# 基础库
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
import joblib

# 第三方库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

# sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    GridSearchCV,
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    precision_recall_curve,
    average_precision_score,
    recall_score,
    confusion_matrix,
    roc_curve,
)
from PIL import Image, ImageOps, ImageDraw
from statsmodels.nonparametric.smoothers_lowess import lowess
# EBM库
try:
    from interpret.glassbox import ExplainableBoostingClassifier
except ImportError:
    ExplainableBoostingClassifier = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

In [ ]:
# 1.2 全局随机种子 + 数据读取 + 训练/测试划分 + 输出目录

RANDOM_STATE = 42
TEST_SIZE = 0.2

DATA_PATH = Path("data.csv")
ID_COL = "ID"
TARGET_COL = "BHR"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# 读取数据
df = pd.read_csv(DATA_PATH)
print("原始数据集形状:", df.shape)

# 构造 X / y
feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int).copy()

# 分层划分训练集/测试集（8:2）
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# 重置索引，后面做 OOF、拼接结果时更整齐
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, 阳性率: {y_train.mean():.4f}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}, 阳性率: {y_test.mean():.4f}")
print(f"阳性率差值(train - test): {(y_train.mean() - y_test.mean()):.4f}")

# 输出目录，每个章节都需要一个单独的输出目录，方便后续整理结果
OUTPUT_ROOT = Path("ebm_output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CURRENT_SECTION = None
SECTION_DIR = None

def set_section(section_num):
    global CURRENT_SECTION, SECTION_DIR
    CURRENT_SECTION = section_num
    SECTION_DIR = OUTPUT_ROOT / f"section{section_num}"
    SECTION_DIR.mkdir(parents=True, exist_ok=True)
    return SECTION_DIR

# 默认先切到第一章输出目录
set_section(1)
print("当前输出目录:", SECTION_DIR.resolve())

In [ ]:
# 1.3.1 绘制分箱堆叠柱状图函数

def plot_stacked_bhr_by_bins(
    df,
    target="BHR",
    n_bins=20,
    cols=None,
    min_count_per_bin=1,
    binning="equal_width",
    max_categories=20,
    value_label_maps=None,
    combine=True,
    ncols=None,
    annotate_p1=True,
    annotate_as_percent=True,
    annotate_decimals=0,
    annotate_position="boundary",
    show_bin_n_in_label=False,
    save_path=None,
    dpi=300,
    bbox_inches="tight",
):
    if value_label_maps is None:
        value_label_maps = {}

    if cols is None:
        cols = [c for c in df.columns if c != target]
    else:
        cols = [c for c in cols if c != target]
    
    def _fmt_val(a):
        return f"{a:.2f}"
    # 区间标注名称
    def _make_interval_label(left, right, n=None):
        lab = f"[{_fmt_val(left)}, {_fmt_val(right)}]"
        if show_bin_n_in_label and n is not None:
            lab += f" (n={int(n)})"
        return lab
    # 分箱概率映射
    def _ct_to_prop(ct):
        ct = ct.reindex(columns=[1, 0], fill_value=0)
        total = ct.sum(axis=1)
        keep = total >= min_count_per_bin
        ct = ct.loc[keep]
        total = total.loc[keep]
        prop = ct.div(total, axis=0).fillna(0.0)
        return prop, total

    def _prepare_one_feature(col):
        sub = df[[col, target]].copy()

        nunique = int(sub[col].nunique(dropna=True))
        is_object_like = pd.api.types.is_object_dtype(sub[col]) or pd.api.types.is_categorical_dtype(sub[col])
        is_bool_like = pd.api.types.is_bool_dtype(sub[col])
        treat_as_categorical = is_object_like or is_bool_like or (nunique <= max_categories)

        if treat_as_categorical:
            ct = pd.crosstab(sub[col], sub[target])
            prop, total = _ct_to_prop(ct)

            mapper = value_label_maps.get(col, {})
            xlabels = [str(mapper.get(k, k)) for k in prop.index.tolist()]
            if show_bin_n_in_label:
                xlabels = [f"{lab} (n={int(total.loc[idx])})" for lab, idx in zip(xlabels, prop.index.tolist())]

            return {"col": col, "prop": prop, "xlabels": xlabels, "mode": "categorical"}

        x = pd.to_numeric(sub[col], errors="coerce")
        sub = sub.loc[x.notna()].copy()
        x = x.loc[x.notna()].astype(float)

        if binning == "equal_count":
            x_sorted = x.sort_values(kind="mergesort")
            sorted_idx = x_sorted.index.to_numpy()
            n = len(sorted_idx)
            bins_eff = min(n_bins, n)
            chunks = np.array_split(np.arange(n), bins_eff)

            bin_id = pd.Series(index=sorted_idx, dtype=int)
            for b, pos in enumerate(chunks):
                bin_id.loc[sorted_idx[pos]] = b

            sub["bin_id"] = bin_id.loc[sub.index].astype(int)
            sub["_x_"] = x.values

            ranges = sub.groupby("bin_id")["_x_"].agg(["min", "max", "count"]).reset_index()
            label_map = {
                int(r["bin_id"]): _make_interval_label(r["min"], r["max"], n=r["count"])
                for _, r in ranges.iterrows()
            }

            ct = pd.crosstab(sub["bin_id"], sub[target]).reindex(range(bins_eff), fill_value=0)
            prop, total = _ct_to_prop(ct)
            xlabels = [label_map[int(i)] for i in prop.index.tolist()]

            return {"col": col, "prop": prop, "xlabels": xlabels, "mode": "equal_count"}

        else:
            x_min = float(x.min())
            x_max = float(x.max())
            if x_min == x_max:
                eps = 1e-6 if x_min == 0 else abs(x_min) * 1e-6
                x_min -= eps
                x_max += eps

            edges = np.linspace(x_min, x_max, n_bins + 1)
            sub["bin"] = pd.cut(x, bins=edges, right=True, include_lowest=True)

            ct = pd.crosstab(sub["bin"], sub[target])
            prop, total = _ct_to_prop(ct)

            xlabels = []
            for idx in prop.index.tolist():
                n_in = int(total.loc[idx]) if show_bin_n_in_label else None
                xlabels.append(_make_interval_label(idx.left, idx.right, n=n_in))

            return {"col": col, "prop": prop, "xlabels": xlabels, "mode": "equal_width"}

    prepared = [_prepare_one_feature(col) for col in cols]

    n = len(prepared)
    if combine:
        ncols_eff = ncols if ncols is not None else int(np.ceil(np.sqrt(n)))
        nrows_eff = int(np.ceil(n / ncols_eff))
    else:
        ncols_eff = 1
        nrows_eff = n

    max_bins = max(len(d["xlabels"]) for d in prepared)

    fig_w = (4.6 + 0.08 * min(max_bins, 35)) * ncols_eff
    fig_h = (4.2 + 0.55) * nrows_eff

    fig, axes = plt.subplots(nrows=nrows_eff, ncols=ncols_eff, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(nrows_eff, ncols_eff)

    for i in range(nrows_eff * ncols_eff):
        r, c = divmod(i, ncols_eff)
        ax = axes[r, c]

        if i >= n:
            ax.axis("off")
            continue

        d = prepared[i]
        prop = d["prop"]
        xlabels = d["xlabels"]
        col = d["col"]

        x_pos = np.arange(len(prop))
        p1 = prop[1].values
        p0 = prop[0].values

        ax.bar(x_pos, p1, label=f"{target}=1")
        ax.bar(x_pos, p0, bottom=p1, label=f"{target}=0")

        ax.set_xticks(x_pos)

        k = len(xlabels)
        if k <= 8:
            xtick_fs, rot = 9, 35
        elif k <= 14:
            xtick_fs, rot = 8, 45
        elif k <= 22:
            xtick_fs, rot = 7, 55
        else:
            xtick_fs, rot = 6, 65

        ax.set_xticklabels(xlabels, rotation=rot, ha="right")
        ax.tick_params(axis="x", labelsize=xtick_fs, pad=10)
        ax.tick_params(axis="y", labelsize=9)

        ax.set_ylim(0, 1.0)
        ax.set_title(col, pad=6)

        if annotate_p1:
            for j, val in enumerate(p1):
                txt = f"{val*100:.{annotate_decimals}f}%" if annotate_as_percent else f"{val:.{annotate_decimals}f}"
                y = val if annotate_position == "boundary" else (val / 2.0)
                y = min(y + 0.015, 0.98)
                ax.text(j, y, txt, ha="center", va="bottom", fontsize=8, clip_on=True)

        ax.margins(x=0.01)

    axes[0, 0].legend(loc="upper right")

    if max_bins <= 10:
        hspace = 0.55
        bottom = 0.16
    elif max_bins <= 18:
        hspace = 0.75
        bottom = 0.20
    else:
        hspace = 0.95
        bottom = 0.24

    fig.subplots_adjust(
        left=0.06, right=0.99,
        top=0.94, bottom=bottom,
        wspace=0.28, hspace=hspace
    )

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches=bbox_inches)
        print(f"[Saved] {save_path.resolve()}")

    plt.show()
    return fig, axes

In [ ]:
# 1.3.1.1 调用上述函数

set_section(1)

# 等宽分箱（箱宽固定，人数不固定）
plot_stacked_bhr_by_bins(
    df=df,
    target=TARGET_COL,
    n_bins=20,
    cols=feature_cols,
    binning="equal_width",
    show_bin_n_in_label=True,
    save_path=SECTION_DIR / "stacked_bhr_equal_width.png"
)

# 等频分箱（箱宽不固定，人数尽量接近）
plot_stacked_bhr_by_bins(df=df,
                         target=TARGET_COL,
                         n_bins=20,
                         cols=feature_cols,
                         binning="equal_count",
                         show_bin_n_in_label=True,
                         save_path=SECTION_DIR / "stacked_bhr_equal_count.png")

#单独调用
# plot_stacked_bhr_by_bins(
#     df=df,
#     target="BHR",
#     cols=["Age"],
#     n_bins=20,
#     binning="equal_width",
#     show_bin_n_in_label=True,
#     save_path=SECTION_DIR /"age_stacked_bar.png"
# )

In [ ]:
# 1.3.2 生成描述性统计表并保存

set_section(1)

desc_rows = []

for col in feature_cols:
    sub = df[[col, TARGET_COL]].copy()
    n_unique = sub[col].nunique(dropna=True)
    #判断分类变量
    is_cat = (
        pd.api.types.is_object_dtype(sub[col]) or
        pd.api.types.is_categorical_dtype(sub[col]) or
        pd.api.types.is_bool_dtype(sub[col]) or
        n_unique <= 5
    )

    # 连续变量
    if not is_cat:
        g_all = sub[col].dropna()
        g0 = sub.loc[sub[TARGET_COL] == 0, col].dropna()
        g1 = sub.loc[sub[TARGET_COL] == 1, col].dropna()

        p_value = stats.mannwhitneyu(g0, g1, alternative="two-sided").pvalue

        desc_rows.append({
            "variable": col,
            "type": "continuous",
            "level": "",
            "overall_n": int(g_all.shape[0]),
            "overall_mean": round(g_all.mean(), 4),
            "overall_var": round(g_all.var(), 4),
            "overall_q1": round(g_all.quantile(0.25), 4),
            "overall_median": round(g_all.median(), 4),
            "overall_q3": round(g_all.quantile(0.75), 4),
            "BHR0_n": int(g0.shape[0]),
            "BHR0_mean": round(g0.mean(), 4),
            "BHR0_var": round(g0.var(), 4),
            "BHR0_q1": round(g0.quantile(0.25), 4),
            "BHR0_median": round(g0.median(), 4),
            "BHR0_q3": round(g0.quantile(0.75), 4),
            "BHR1_n": int(g1.shape[0]),
            "BHR1_mean": round(g1.mean(), 4),
            "BHR1_var": round(g1.var(), 4),
            "BHR1_q1": round(g1.quantile(0.25), 4),
            "BHR1_median": round(g1.median(), 4),
            "BHR1_q3": round(g1.quantile(0.75), 4),
            "p_value": round(p_value, 6)
        })

    # 分类变量
    else:
        ct = pd.crosstab(sub[col], sub[TARGET_COL])

        if ct.shape == (2, 2):
            p_value = stats.fisher_exact(ct)[1]
        else:
            p_value = stats.chi2_contingency(ct)[1]

        total_n = len(sub)
        total0 = (sub[TARGET_COL] == 0).sum()
        total1 = (sub[TARGET_COL] == 1).sum()

        for level in sub[col].dropna().unique():
            n_all = (sub[col] == level).sum()
            n0 = ((sub[col] == level) & (sub[TARGET_COL] == 0)).sum()
            n1 = ((sub[col] == level) & (sub[TARGET_COL] == 1)).sum()

            desc_rows.append({
                "variable": col,
                "type": "categorical",
                "level": level,
                "overall_n": int(n_all),
                "overall_pct": round(n_all / total_n * 100, 2),
                "BHR0_n": int(n0),
                "BHR0_pct": round(n0 / total0 * 100, 2),
                "BHR1_n": int(n1),
                "BHR1_pct": round(n1 / total1 * 100, 2),
                "p_value": round(p_value, 6)
            })

descriptive_table_df = pd.DataFrame(desc_rows)
descriptive_table_df.to_csv(SECTION_DIR / "descriptive_statistics_table.csv", index=False, encoding="utf-8-sig")

print("描述性统计表已保存：", SECTION_DIR / "descriptive_statistics_table.csv")
descriptive_table_df.head(20)

In [ ]:
# 2.1.1 准备第二章输出目录

set_section(2)
print("当前输出目录:", SECTION_DIR.resolve())

In [ ]:
# 2.1.2 计算相关矩阵，绘制Spearman热力图，筛选高相关特征对

import seaborn as sns

corr_df = X_train[feature_cols].copy()

# Spearman相关矩阵
corr_matrix = corr_df.corr(method="spearman")
corr_matrix.to_csv(SECTION_DIR / "corr_matrix_spearman.csv", encoding="utf-8-sig")

# 绘制带具体数值的热力图
plt.figure(figsize=(13, 11), dpi=300)
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 8}
)
plt.title("Spearman Correlation Heatmap", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(SECTION_DIR / "corr_heatmap_spearman.png", dpi=300, bbox_inches="tight")
plt.show()

# 筛选高相关特征对
corr_threshold = 0.70

high_corr_rows = []
cols = corr_matrix.columns.tolist()

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) >= corr_threshold:
            high_corr_rows.append({
                "feature_1": cols[i],
                "feature_2": cols[j],
                "spearman_r": round(r, 4),
                "abs_r": round(abs(r), 4)
            })

high_corr_pairs_df = pd.DataFrame(high_corr_rows).sort_values(
    "abs_r", ascending=False
).reset_index(drop=True)

high_corr_pairs_df.to_csv(SECTION_DIR / "high_corr_pairs.csv", index=False, encoding="utf-8-sig")

print(f"|r| >= {corr_threshold} 的高相关特征对数量: {len(high_corr_pairs_df)}")
high_corr_pairs_df.head(20)

In [ ]:
# 2.2.1 基于EBM-main（无交互）的特征重要性函数

def ebm_repeated_feature_importance(
    X,
    y,
    feature_list,
    n_splits=5,
    n_repeats=10,#10次重复，提升稳定性
    random_state=RANDOM_STATE,#统一随机
    top_k=5,#这里选5个是因为用50%的特征，认为进入前一半就很重要
    save_dir=None
):
    X_use = X[feature_list].copy().reset_index(drop=True)
    y_use = y.reset_index(drop=True)

    all_rows = []

    for rep in range(n_repeats):
        cv = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state + rep
        )

        for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_use, y_use), start=1):
            X_tr = X_use.iloc[tr_idx]
            y_tr = y_use.iloc[tr_idx]

            ebm = ExplainableBoostingClassifier(
                interactions=0,
                random_state=random_state + rep
            )
            ebm.fit(X_tr, y_tr)

            global_exp = ebm.explain_global()
            term_names = global_exp.data()["names"]
            term_scores = global_exp.data()["scores"]

            for name, score in zip(term_names, term_scores):
                if name in feature_list:
                    all_rows.append({
                        "repeat": rep + 1,
                        "fold": fold_idx,
                        "feature": name,
                        "importance": float(score)
                    })

    importance_long_df = pd.DataFrame(all_rows)

    importance_summary_df = (
        importance_long_df
        .groupby("feature", as_index=False)["importance"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "importance_mean", "std": "importance_std"})
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )

    # 统计每次repeat-fold的top_k出现频数
    topk_rows = []
    for (rep, fold), sub_df in importance_long_df.groupby(["repeat", "fold"]):
        top_features = (
            sub_df.sort_values("importance", ascending=False)
            .head(top_k)["feature"]
            .tolist()
        )
        for f in top_features:
            topk_rows.append({"feature": f, "topk_hit": 1})

    topk_hit_df = (
        pd.DataFrame(topk_rows)
        .groupby("feature", as_index=False)["topk_hit"]
        .sum()
        .sort_values("topk_hit", ascending=False)
    )

    importance_summary_df = importance_summary_df.merge(
        topk_hit_df,
        on="feature",
        how="left"
    ).fillna({"topk_hit": 0})

    importance_summary_df["topk_hit"] = importance_summary_df["topk_hit"].astype(int)

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        importance_long_df.to_csv(
            save_dir / "ebm_feature_importance_long.csv",
            index=False,
            encoding="utf-8-sig"
        )
        importance_summary_df.to_csv(
            save_dir / "ebm_feature_importance_summary.csv",
            index=False,
            encoding="utf-8-sig"
        )

        # 画图
        plot_df = importance_summary_df.sort_values("importance_mean", ascending=True)

        plt.figure(figsize=(8, max(4, len(plot_df) * 0.45)), dpi=300)
        plt.barh(
            plot_df["feature"],
            plot_df["importance_mean"],
            xerr=plot_df["importance_std"].fillna(0).values,
            capsize=3
        )
        plt.xlabel("Mean importance")
        plt.ylabel("Feature")
        plt.title("EBM-main Repeated Feature Importance")
        plt.tight_layout()
        plt.savefig(save_dir / "ebm_feature_importance_barplot.png", dpi=300, bbox_inches="tight")
        plt.show()

    return importance_long_df, importance_summary_df

In [ ]:
# 2.2.2 调用上述函数并评估特征重要性

set_section(2)

importance_feature_list = [
    "EOS",
    "FVC%",
    "FEV1%",
    "FEV1/FVC",
    "PEF%",
    "MMEF%",
    "MEF75%",
    "MEF50%",
    "MEF25%",
    "WBC"
]

importance_save_dir = SECTION_DIR 

importance_long_df, importance_summary_df = ebm_repeated_feature_importance(
    X=X_train,
    y=y_train,
    feature_list=importance_feature_list,
    n_splits=5,
    n_repeats=10,
    random_state=RANDOM_STATE,
    top_k=5,
    save_dir=importance_save_dir
)

print("EBM-main 特征重要性汇总：")
display(importance_summary_df)

top5_features = importance_summary_df["feature"].head(5).tolist()
print("Top 5 特征：", top5_features)

In [ ]:
# 2.3 准备三组特征清单
"""其实就是准备一下，有几个争议的特征，万一后面需要对比，
但其实如果要多特征组对比再去观察，需要的配置要求比较高，
而且这里准备好的话，后面如果改特征组，修改起来比较简单
"""
FEATURE_GROUP_1 = [
    "Male=0",
    "Age",
    "BMI",
    "Height",
    "EOS",
    "FVC%",
    "FEV1/FVC",
    "MMEF%",
    "MEF50%"
]

FEATURE_GROUP_2 = [
    "Male=0",
    "Age",
    "BMI",
    "Height",
    "EOS",
    "FEV1/FVC",
    "MMEF%",
    "MEF50%"
]

FEATURE_GROUP_3 = [
    "Age",
    "BMI",
    "EOS",
    "FVC%",
    "FEV1/FVC",
    "MMEF%",
    "MEF50%"
]

feature_group_dict = {
    "Group1_9vars": FEATURE_GROUP_1,
    "Group2_8vars_noHeight": FEATURE_GROUP_2,
    "Group3_7vars_noHeight_noSex": FEATURE_GROUP_3,
}

for k, v in feature_group_dict.items():
    print(f"{k}: {len(v)} 个特征")
    print(v)
    print("-" * 60)

In [ ]:
# 3.1.1 固定第三章输出目录

set_section(3)
print("当前输出目录:", SECTION_DIR.resolve())

In [ ]:
# 3.1.2 固定后续入模特征，子集数据，统一评估工具

# 从这里，上面准备的特征清单的优势就比较好了，后面想改直接改这一行
selected_features = FEATURE_GROUP_1.copy()

X_train_sel = X_train[selected_features].copy().reset_index(drop=True)
X_test_sel = X_test[selected_features].copy().reset_index(drop=True)
y_train_sel = y_train.reset_index(drop=True)
y_test_sel = y_test.reset_index(drop=True)

print("当前入模特征：", selected_features)
print("X_train_sel:", X_train_sel.shape)
print("X_test_sel :", X_test_sel.shape)

#清理掉字符串里的%，/，\\和空格，避免后续文件命名、图表标题等出现问题
def safe_name(s):
    return str(s).replace("/", "_").replace("\\", "_").replace("%", "pct").replace(" ", "_")

#后面讨论了一下，医学上偏爱youden
def find_best_threshold(y_true, y_prob, strategy="youden"):
    thresholds = np.unique(y_prob)

    best_thr = 0.5
    best_score = -np.inf
    best_info = {}

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0.0
        youden = sensitivity + specificity - 1

        score = f1 if strategy == "f1" else youden

        if score > best_score:
            best_score = score
            best_thr = thr
            best_info = {
                "f1": f1,
                "youden": youden,
                "precision": precision,
                "sensitivity": sensitivity,
                "specificity": specificity
            }

    return float(best_thr), best_info


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    youden = sensitivity + specificity - 1  

    return {
        "auc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "youden": youden,
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def plot_roc_curves(y_true, pred_prob_dict, save_path=None, title="ROC Curves"):
    plt.figure(figsize=(7, 6), dpi=300)

    for model_name, y_prob in pred_prob_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_value = roc_auc_score(y_true, y_prob)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right", frameon=False)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()


def plot_confusion_matrix_single(y_true, y_prob, threshold=0.5, title=None, save_path=None):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(4.5, 4), dpi=300)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"]
    )
    plt.title(title if title is not None else f"Confusion Matrix (thr={threshold:.3f})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# 3.2.1 设定 Benchmark 组参数及参数网格

N_JOBS = max(1, (os.cpu_count() or 2) - 1)#电脑太拉，留一个核给系统
print("[Info] N_JOBS =", N_JOBS)
#万一有不能用的参数就跳过去
def filter_param_grid(estimator, param_grid):
    valid = set(estimator.get_params(deep=True).keys())
    return {k: v for k, v in param_grid.items() if k in valid}
#计算网格搜索的参数组合总数
def grid_size(param_grid):
    n = 1
    for v in param_grid.values():
        n *= len(v)
    return n

def make_logreg_pipeline(penalty, solver, max_iter):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty=penalty,
            solver=solver,
            max_iter=max_iter,
            random_state=RANDOM_STATE
        ))
    ])

benchmark_models = {}
benchmark_param_grids = {}

# LR
lr_pipeline = make_logreg_pipeline(penalty="l2", solver="lbfgs", max_iter=4000)
#一般来讲这两个特征就算是最重要的了，c控制正则化强度，class_weight控制类别不平衡的处理方式
# solver一般影响的收敛速度和稳定性，我们的模型慢一点也没关系,但后面看到class_weight=balanced会影响校准曲线
# 就会自动处理类别不平衡，所以这里就不用管class_weight了
lr_param_grid = filter_param_grid(lr_pipeline, {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__class_weight": [None],
})

benchmark_models["LR"] = lr_pipeline
benchmark_param_grids["LR"] = lr_param_grid

# LASSO
lasso_pipeline = make_logreg_pipeline(
    penalty="l1",
    solver="saga",
    max_iter=8000
)
#对于lasso而言，最重要的就是一阶罚项，c就是用来压缩的，最重要。所以solver什么的直接写死就行
#c越小，越多特征被压缩，其余没那么重要，再说网格画太大也跑不完
lasso_param_grid = filter_param_grid(lasso_pipeline, {
    "clf__C": [0.05, 0.1, 0.5, 1.0],
    "clf__class_weight": [None],
})

benchmark_models["LASSO"] = lasso_pipeline
benchmark_param_grids["LASSO"] = lasso_param_grid

# RF
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
rf_param_grid = filter_param_grid(rf_model, {
    "n_estimators": [300, 600],
    "max_depth": [None, 6, 10],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt"],
    "class_weight": [None, "balanced"],
})

benchmark_models["RF"] = rf_model
benchmark_param_grids["RF"] = rf_param_grid

# XGB
if XGBClassifier is not None:
    xgb_model = XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        eval_metric="logloss",
        tree_method="hist",
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
    )
    #learning_rate控制每棵树的贡献度，最重要，max-depth控制树的深度，越深越复杂
    #n_estimators控制树的数量，越小越快
    xgb_param_grid = filter_param_grid(
        xgb_model, {
            "n_estimators": [300, 600],
            "learning_rate": [0.03, 0.1],
            "max_depth": [3, 4],
            "min_child_weight": [1, 5],
        })

    benchmark_models["XGB"] = xgb_model
    benchmark_param_grids["XGB"] = xgb_param_grid

# LGBM
if lgb is not None:
    lgbm_model = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=-1,
        subsample=0.8,
        colsample_bytree=0.8,
    )
    #基本和上面的XGB的参数类似，但多了一个class_weight，控制类别不平衡的处理方式，和num_leaves，控制树的复杂度，越大越复杂
    lgbm_param_grid = filter_param_grid(
        lgbm_model, {
            "n_estimators": [300, 600],
            "learning_rate": [0.03, 0.1],
            "num_leaves": [15, 31],
            "min_child_samples": [10, 30],
            "class_weight": [None, "balanced"],
        })

    benchmark_models["LGBM"] = lgbm_model
    benchmark_param_grids["LGBM"] = lgbm_param_grid

# EBM_main
if ExplainableBoostingClassifier is not None:
    ebm_main_model = ExplainableBoostingClassifier(
        random_state=RANDOM_STATE,
        interactions=0,
    )
    ebm_main_param_grid = filter_param_grid(ebm_main_model, {
        "max_bins": [64, 128],
        "min_samples_leaf": [1, 5],
    })

    benchmark_models["EBM_main"] = ebm_main_model
    benchmark_param_grids["EBM_main"] = ebm_main_param_grid

print("[Info] Benchmark models ready:", list(benchmark_models.keys()))
for name in benchmark_models:
    print(
        f"[Grid size] {name}: {grid_size(benchmark_param_grids[name])} combinations"
    )

In [ ]:
# 3.2.2 设定 EBM 消融组参数及参数网格

ablation_models = {}
ablation_param_grids = {}

if ExplainableBoostingClassifier is None:
    warnings.warn("[Warn] interpret/EBM 不可用，跳过 3.2.2 的 EBM 消融组。")
else:
    ebm_inter_model = ExplainableBoostingClassifier(
        random_state=RANDOM_STATE,
        interactions=10,
    )

    ebm_inter_param_grid = filter_param_grid(ebm_inter_model, {
        "max_bins": [64, 128],
        "min_samples_leaf": [1, 5],
    })

    ablation_models["EBM_inter"] = ebm_inter_model
    ablation_param_grids["EBM_inter"] = ebm_inter_param_grid

    print("Added EBM ablation variants:", list(ablation_models.keys()))
    for name in ablation_models:
        print(f"[Grid size] {name}: {grid_size(ablation_param_grids[name])} combinations")

In [ ]:
# 3.3.1 对每个候选模型，在训练集上用固定CV方案搜索最优超参数（5折）
# 调参单个五折即可，没有必要为了稳定性开发多次重复
# 这个cell调参，下个cell找阈值
set_section(3)

all_models = {}
all_models.update(benchmark_models)
all_models.update(ablation_models)

all_param_grids = {}
all_param_grids.update(benchmark_param_grids)
all_param_grids.update(ablation_param_grids)

cv_tune = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

best_estimators = {}
best_params_dict = {}
tuning_rows = []

for model_name, model in all_models.items():
    print(f"正在调参: {model_name}")

    grid = GridSearchCV(
        estimator=model,
        param_grid=all_param_grids[model_name],
        scoring={
            "auc": "roc_auc",
            "auprc": "average_precision"
        },
        refit="auc",
        cv=cv_tune,
        n_jobs=-1,
        verbose=0,
        return_train_score=False
    )

    grid.fit(X_train_sel, y_train_sel)

    best_estimators[model_name] = grid.best_estimator_
    best_params_dict[model_name] = grid.best_params_

    best_idx = grid.best_index_
    tuning_rows.append({
        "model": model_name,
        "best_auc_cv": round(grid.cv_results_["mean_test_auc"][best_idx], 4),
        "best_auprc_cv": round(grid.cv_results_["mean_test_auprc"][best_idx], 4),
        "best_params": str(grid.best_params_)
    })

tuning_summary_df = pd.DataFrame(tuning_rows).sort_values(
    "best_auc_cv", ascending=False
).reset_index(drop=True)

tuning_summary_df.to_csv(
    SECTION_DIR / "tuning_summary.csv",
    index=False,
    encoding="utf-8-sig"
)
#保存最优参数万一后面调用
with open(SECTION_DIR / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params_dict, f, ensure_ascii=False, indent=2)

print("调参完成。")
display(tuning_summary_df)

In [ ]:
# 3.3.2 5折×20次 OOF 阈值学习与内部稳定性分析

repeat_rows = []
final_thresholds = {}
final_youden = {}
oof_pred_dict_by_model = {}

n_repeats = 20
n_splits = 5
threshold_strategy = "youden"

for model_name, fitted_model in best_estimators.items():
    print(f"正在进行重复 OOF: {model_name}")

    model_repeat_rows = []
    model_oof_list = []
    threshold_list = []
    youden_list = []

    for rep in range(n_repeats):
        cv_oof = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE + rep
        )

        oof_prob = np.zeros(len(X_train_sel))

        for fold_idx, (tr_idx, va_idx) in enumerate(cv_oof.split(X_train_sel, y_train_sel), start=1):
            X_tr = X_train_sel.iloc[tr_idx]
            X_va = X_train_sel.iloc[va_idx]
            y_tr = y_train_sel.iloc[tr_idx]

            model = clone(fitted_model)
            model.fit(X_tr, y_tr)

            oof_prob[va_idx] = model.predict_proba(X_va)[:, 1]

        best_thr, thr_info = find_best_threshold(
            y_true=y_train_sel.values,
            y_prob=oof_prob,
            strategy=threshold_strategy
        )

        met = compute_metrics(
            y_true=y_train_sel.values,
            y_prob=oof_prob,
            threshold=best_thr
        )

        row = {
            "model": model_name,
            "repeat": rep + 1,
            "threshold": round(best_thr, 6),
            "threshold_strategy": threshold_strategy,
            "youden_index": round(thr_info.get("youden", 0), 6) if thr_info else 0,
            "auc": round(met["auc"], 6),
            "auprc": round(met["auprc"], 6),
            "accuracy": round(met["accuracy"], 6),
            "balanced_accuracy": round(met["balanced_accuracy"], 6),
            "precision": round(met["precision"], 6),
            "sensitivity": round(met["sensitivity"], 6),
            "specificity": round(met["specificity"], 6),
            "f1": round(met["f1"], 6)
        }

        model_repeat_rows.append(row)
        model_oof_list.append(oof_prob.copy())
        threshold_list.append(best_thr)
        youden_list.append(thr_info.get("youden", 0) if thr_info else 0)

    model_repeat_df = pd.DataFrame(model_repeat_rows)
    repeat_rows.append(model_repeat_df)

    final_thresholds[model_name] = float(np.median(threshold_list))
    final_youden[model_name] = float(np.median(youden_list))
    oof_pred_dict_by_model[model_name] = model_oof_list

repeated_metrics_df = pd.concat(repeat_rows, axis=0).reset_index(drop=True)

summary_rows = []
for model_name, sub_df in repeated_metrics_df.groupby("model"):
    summary_rows.append({
        "model": model_name,
        "youden_index_mean": round(sub_df["youden_index"].mean(), 4),
        "youden_index_std": round(sub_df["youden_index"].std(ddof=1), 4),
        "auc_mean": round(sub_df["auc"].mean(), 4),
        "auc_std": round(sub_df["auc"].std(ddof=1), 4),
        "auprc_mean": round(sub_df["auprc"].mean(), 4),
        "auprc_std": round(sub_df["auprc"].std(ddof=1), 4),
        "accuracy_mean": round(sub_df["accuracy"].mean(), 4),
        "accuracy_std": round(sub_df["accuracy"].std(ddof=1), 4),
        "balanced_accuracy_mean": round(sub_df["balanced_accuracy"].mean(), 4),
        "balanced_accuracy_std": round(sub_df["balanced_accuracy"].std(ddof=1), 4),
        "precision_mean": round(sub_df["precision"].mean(), 4),
        "precision_std": round(sub_df["precision"].std(ddof=1), 4),
        "sensitivity_mean": round(sub_df["sensitivity"].mean(), 4),
        "sensitivity_std": round(sub_df["sensitivity"].std(ddof=1), 4),
        "specificity_mean": round(sub_df["specificity"].mean(), 4),
        "specificity_std": round(sub_df["specificity"].std(ddof=1), 4),
        "f1_mean": round(sub_df["f1"].mean(), 4),
        "f1_std": round(sub_df["f1"].std(ddof=1), 4),
        "threshold_median": round(sub_df["threshold"].median(), 6),
        "threshold_mean": round(sub_df["threshold"].mean(), 6),
        "threshold_std": round(sub_df["threshold"].std(ddof=1), 6)
    })

repeated_summary_df = pd.DataFrame(summary_rows).sort_values(
    "auc_mean", ascending=False
).reset_index(drop=True)

repeated_metrics_df.to_csv(
    SECTION_DIR / "repeated_oof_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

repeated_summary_df.to_csv(
    SECTION_DIR / "repeated_oof_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

final_threshold_df = pd.DataFrame({
    "model": list(final_thresholds.keys()),
    "final_threshold_median": [round(final_thresholds[k], 6) for k in final_thresholds.keys()],
    "final_youden_median": [round(final_youden[k], 6) for k in final_thresholds.keys()]
}).sort_values("model").reset_index(drop=True)

final_threshold_df.to_csv(
    SECTION_DIR / "final_thresholds_from_repeated_oof.csv",
    index=False,
    encoding="utf-8-sig"
)
base_params = fitted_model.get_params()
model_class = fitted_model.__class__
print("Repeated OOF 完成。")
display(repeated_summary_df)
display(final_threshold_df)

In [ ]:
# 3.3.3 用最优参数和中位数阈值在完整训练集上重训最终模型

final_models = {}
final_train_pred_prob_dict = {}
final_model_train_metrics_rows = []

for model_name, best_model in best_estimators.items():
    print(f"正在重训最终模型: {model_name}")

    model = clone(best_model)
    model.fit(X_train_sel, y_train_sel)

    final_models[model_name] = model

    train_prob = model.predict_proba(X_train_sel)[:, 1]
    final_train_pred_prob_dict[model_name] = train_prob

    thr = final_thresholds[model_name]

    met = compute_metrics(
        y_true=y_train_sel.values,
        y_prob=train_prob,
        threshold=thr
    )

    final_model_train_metrics_rows.append({
        "model": model_name,
        "threshold": round(thr, 6),
        "final_youden_median": round(final_youden[model_name], 6),
        "auc_train_full": round(met["auc"], 6),
        "auprc_train_full": round(met["auprc"], 6),
        "accuracy_train_full": round(met["accuracy"], 6),
        "balanced_accuracy_train_full": round(met["balanced_accuracy"], 6),
        "precision_train_full": round(met["precision"], 6),
        "sensitivity_train_full": round(met["sensitivity"], 6),
        "specificity_train_full": round(met["specificity"], 6),
        "f1_train_full": round(met["f1"], 6)
    })

train_pred_prob_df = pd.DataFrame({
    "y_true": y_train_sel.values
})

for model_name, prob in final_train_pred_prob_dict.items():
    train_pred_prob_df[model_name] = prob

train_pred_prob_df.to_csv(
    SECTION_DIR / "final_model_train_pred_prob.csv",
    index=False,
    encoding="utf-8-sig"
)

final_threshold_df = pd.DataFrame({
    "model": list(final_thresholds.keys()),
    "final_threshold": [round(final_thresholds[k], 6) for k in final_thresholds.keys()],
    "final_youden_median": [round(final_youden[k], 6) for k in final_thresholds.keys()]
}).sort_values("model").reset_index(drop=True)


final_model_train_metrics_df = pd.DataFrame(final_model_train_metrics_rows).sort_values(
    "auc_train_full", ascending=False
).reset_index(drop=True)

final_model_train_metrics_df.to_csv(
    SECTION_DIR / "final_model_train_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

final_model_specs = {}
for model_name, model in final_models.items():
    params = model.get_params()
    params_str = {k: str(v) for k, v in params.items()}
    final_model_specs[model_name] = params_str

with open(SECTION_DIR / "final_model_specs.json", "w", encoding="utf-8") as f:
    json.dump(final_model_specs, f, ensure_ascii=False, indent=2)

print("最终模型和最终阈值已锁定。")
display(final_threshold_df)
display(final_model_train_metrics_df)

In [ ]:
# 3.4 保存最终模型及部署所需信息
# 考虑后续会开发计算器，所以需要这些
save_dir = SECTION_DIR / "saved_final_models"
save_dir.mkdir(parents=True, exist_ok=True)


def _to_jsonable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(v) for v in obj]

    return str(obj)


# 保存特征信息
feature_info = pd.DataFrame({
    "feature": X_train_sel.columns,
    "dtype": [str(X_train_sel[c].dtype) for c in X_train_sel.columns],
    "n_missing_train": [int(X_train_sel[c].isna().sum()) for c in X_train_sel.columns],
    "min": [
        float(X_train_sel[c].min()) if pd.api.types.is_numeric_dtype(X_train_sel[c]) else None
        for c in X_train_sel.columns
    ],
    "max": [
        float(X_train_sel[c].max()) if pd.api.types.is_numeric_dtype(X_train_sel[c]) else None
        for c in X_train_sel.columns
    ],
})
feature_info.to_csv(
    save_dir / "feature_info.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(save_dir / "feature_order.json", "w", encoding="utf-8") as f:
    json.dump(list(X_train_sel.columns), f, ensure_ascii=False, indent=2)


# 方便后面取训练集表现
train_metric_map = {}
if "final_model_train_metrics_df" in globals():
    train_metric_map = (
        final_model_train_metrics_df
        .set_index("model")
        .to_dict(orient="index")
    )

model_records = []

for name, model in final_models.items():
    model_file = save_dir / f"{name}.joblib"
    meta_file = save_dir / f"{name}_meta.json"

    # 先保存模型本体
    joblib.dump(model, model_file)

    # 再保存配套元信息
    meta = {
        "model_name": name,
        "model_class": model.__class__.__name__,
        "feature_names": list(X_train_sel.columns),
        "feature_dtypes": {c: str(X_train_sel[c].dtype) for c in X_train_sel.columns},
        "n_features": int(X_train_sel.shape[1]),
        "threshold": float(final_thresholds[name]),
        "best_params": _to_jsonable(best_params_dict.get(name, {})),
        "model_params": _to_jsonable(model.get_params(deep=True)),
        "train_metrics": _to_jsonable(train_metric_map.get(name, {})),
    }

    with open(meta_file, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    model_records.append({
        "model_name": name,
        "model_class": model.__class__.__name__,
        "model_file": model_file.name,
        "meta_file": meta_file.name,
        "threshold": float(final_thresholds[name]),
        "n_features": int(X_train_sel.shape[1]),
    })

model_records_df = pd.DataFrame(model_records).sort_values(
    "model_name"
).reset_index(drop=True)

model_records_df.to_csv(
    save_dir / "saved_models_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

project_meta = {
    "target_name": TARGET_COL,
    "random_state": RANDOM_STATE,
    "selected_features": list(X_train_sel.columns),
    "n_features": int(X_train_sel.shape[1]),
    "train_size": int(len(X_train_sel)),
    "test_size": int(len(X_test_sel)),
    "train_positive_rate": float(y_train_sel.mean()),
    "test_positive_rate": float(y_test_sel.mean()),
    "saved_models": model_records_df["model_name"].tolist(),
}

with open(save_dir / "project_meta.json", "w", encoding="utf-8") as f:
    json.dump(project_meta, f, ensure_ascii=False, indent=2)

display(model_records_df)
print(f"已保存到: {save_dir.resolve()}")


In [ ]:
# 4.1.1 固定第四章输出目录

set_section(4)
print("当前输出目录:", SECTION_DIR.resolve())

In [ ]:
# 4.1.2 测试集一次性评估

test_pred_prob_dict = {}
test_metric_rows = []

for model_name, model in final_models.items():
    print(f"正在评估测试集: {model_name}")

    y_prob = model.predict_proba(X_test_sel)[:, 1]
    thr = final_thresholds[model_name]

    test_pred_prob_dict[model_name] = y_prob

    met = compute_metrics(
        y_true=y_test_sel.values,
        y_prob=y_prob,
        threshold=thr
    )

    test_metric_rows.append({
        "model": model_name,
        "threshold": round(thr, 6),
        "final_youden_median": round(final_youden[model_name], 6),
        "auc_test": round(met["auc"], 6),
        "auprc_test": round(met["auprc"], 6),
        "accuracy_test": round(met["accuracy"], 6),
        "balanced_accuracy_test": round(met["balanced_accuracy"], 6),
        "precision_test": round(met["precision"], 6),
        "sensitivity_test": round(met["sensitivity"], 6),
        "specificity_test": round(met["specificity"], 6),
        "f1_test": round(met["f1"], 6),
        "tn": met["tn"],
        "fp": met["fp"],
        "fn": met["fn"],
        "tp": met["tp"]
    })

test_pred_prob_df = pd.DataFrame({
    "y_true": y_test_sel.values
})

for model_name, y_prob in test_pred_prob_dict.items():
    test_pred_prob_df[model_name] = y_prob

test_pred_prob_df.to_csv(
    SECTION_DIR / "test_pred_prob_all_models.csv",
    index=False,
    encoding="utf-8-sig"
)

test_metrics_df = pd.DataFrame(test_metric_rows).sort_values(
    "auc_test", ascending=False
).reset_index(drop=True)

test_metrics_df.to_csv(
    SECTION_DIR / "test_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

plot_roc_curves(
    y_true=y_test_sel.values,
    pred_prob_dict=test_pred_prob_dict,
    save_path=SECTION_DIR / "test_roc_curves.png",
    title="ROC Curves on Test Set"
)


plt.figure(figsize=(7, 6), dpi=300)

for model_name, y_prob in test_pred_prob_dict.items():
    precision, recall, _ = precision_recall_curve(y_test_sel.values, y_prob)
    auprc_value = average_precision_score(y_test_sel.values, y_prob)
    plt.plot(recall, precision, lw=2, label=f"{model_name} (AUPRC={auprc_value:.4f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curves on Test Set")
plt.legend(loc="lower left", frameon=False)
plt.tight_layout()
plt.savefig(SECTION_DIR / "test_pr_curves.png", dpi=300, bbox_inches="tight")
plt.show()







cm_dir = SECTION_DIR / "test_confusion_matrices"
cm_dir.mkdir(parents=True, exist_ok=True)

for model_name, y_prob in test_pred_prob_dict.items():
    thr = final_thresholds[model_name]

    plot_confusion_matrix_single(
        y_true=y_test_sel.values,
        y_prob=y_prob,
        threshold=thr,
        title=f"{model_name} Test Confusion Matrix",
        save_path=cm_dir / f"{safe_name(model_name)}_test_cm.png"
    )

print("测试集一次性评估完成。")
display(test_metrics_df)

#加一个函数用于后面的拼图

def make_image_grid(
    image_paths,
    out_path,
    nrows=None,
    ncols=None,
    panel_pad=20,
    outer_pad=20,
    bg_color="white",
    keep_order=True,
    add_panel_labels=False
):
    """
    将多张已保存图片拼成一张总图并保存。

    参数
    ----
    image_paths : list
        图片路径列表
    out_path : str or Path
        输出总图路径
    nrows, ncols : int or None
        若不指定，则自动判断布局
    panel_pad : int
        子图之间的间距
    outer_pad : int
        画布四周留白
    bg_color : str
        背景颜色
    keep_order : bool
        是否保持输入顺序
    add_panel_labels : bool
        是否添加 (A), (B), (C) ... 标签
    """
    image_paths = [Path(p) for p in image_paths if Path(p).exists()]
    if len(image_paths) == 0:
        raise ValueError("没有找到可拼接的图片。")

    if not keep_order:
        image_paths = sorted(image_paths)

    n = len(image_paths)

    # 自动布局
    if nrows is None or ncols is None:
        layout_map = {
            1: (1, 1),
            2: (1, 2),
            3: (1, 3),
            4: (2, 2),
            5: (2, 3),
            6: (2, 3),
            7: (2, 4),
            8: (2, 4),
            9: (3, 3),
            10: (3, 4),
            11: (3, 4),
            12: (3, 4),
        }
        if n in layout_map:
            nrows, ncols = layout_map[n]
        else:
            ncols = int(np.ceil(np.sqrt(n)))
            nrows = int(np.ceil(n / ncols))

    # 读取图片
    imgs = [Image.open(p).convert("RGB") for p in image_paths]

    # 统一每个格子的尺寸（按最大宽高）
    cell_w = max(im.width for im in imgs)
    cell_h = max(im.height for im in imgs)

    # 画布尺寸
    canvas_w = outer_pad * 2 + ncols * cell_w + (ncols - 1) * panel_pad
    canvas_h = outer_pad * 2 + nrows * cell_h + (nrows - 1) * panel_pad

    canvas = Image.new("RGB", (canvas_w, canvas_h), color=bg_color)
    draw = ImageDraw.Draw(canvas)

    for i, im in enumerate(imgs):
        r = i // ncols
        c = i % ncols

        x0 = outer_pad + c * (cell_w + panel_pad)
        y0 = outer_pad + r * (cell_h + panel_pad)

        # 保持原图比例，放进 cell
        im_fit = ImageOps.contain(im, (cell_w, cell_h))
        x_img = x0 + (cell_w - im_fit.width) // 2
        y_img = y0 + (cell_h - im_fit.height) // 2

        canvas.paste(im_fit, (x_img, y_img))

        if add_panel_labels:
            label = f"({chr(65 + i)})"
            draw.text((x0 + 8, y0 + 8), label, fill="black")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path, quality=95)
    print("[Saved]", out_path.resolve())
cm_dir = SECTION_DIR / "test_confusion_matrices"

cm_paths = sorted(cm_dir.glob("*.png"))

make_image_grid(
    image_paths=cm_paths,
    out_path=cm_dir / "confusion_matrix_grid.png",
    add_panel_labels=True
)


In [ ]:
# 4.1.3 测试集 95% CI（1000 次 bootstrap）

BOOTSTRAP_N = 1000
N_BOOT = BOOTSTRAP_N
rng = np.random.RandomState(RANDOM_STATE)

boot_rows = []

for model_name, y_prob in test_pred_prob_dict.items():
    print(f"正在bootstrap: {model_name}")

    thr = final_thresholds[model_name]
    y_true = y_test_sel.values
    n = len(y_true)

    valid_boot = 0
    for b in range(N_BOOT):
        idx = rng.choice(np.arange(n), size=n, replace=True)

        y_boot = y_true[idx]
        p_boot = y_prob[idx]

        if len(np.unique(y_boot)) < 2:
            continue

        met = compute_metrics(
            y_true=y_boot,
            y_prob=p_boot,
            threshold=thr
        )

        boot_rows.append({
            "model": model_name,
            "bootstrap_id": b + 1,
            "threshold": round(thr, 6),
            "final_youden_median": round(final_youden[model_name], 6),
            "auc": met["auc"],
            "auprc": met["auprc"],
            "accuracy": met["accuracy"],
            "balanced_accuracy": met["balanced_accuracy"],
            "precision": met["precision"],
            "sensitivity": met["sensitivity"],
            "specificity": met["specificity"],
            "f1": met["f1"]
        })
        valid_boot += 1

    print(f"{model_name}: 有效bootstrap次数 = {valid_boot}")

bootstrap_long_df = pd.DataFrame(boot_rows)
bootstrap_long_df.to_csv(
    SECTION_DIR / "bootstrap_metric_dist_long.csv",
    index=False,
    encoding="utf-8-sig"
)

ci_rows = []
metric_cols = [
    "auc", "auprc", "accuracy", "balanced_accuracy",
    "precision", "sensitivity", "specificity", "f1"
]

for model_name, sub_df in bootstrap_long_df.groupby("model"):
    point_row = test_metrics_df.loc[test_metrics_df["model"] == model_name].iloc[0]

    row = {
        "model": model_name,
        "threshold": round(point_row["threshold"], 6),
        "final_youden_median": round(point_row["final_youden_median"], 6)
    }
    for m in metric_cols:
        row[f"{m}_point"] = point_row[f"{m}_test"]
        row[f"{m}_ci_low"] = round(sub_df[m].quantile(0.025), 4)
        row[f"{m}_ci_high"] = round(sub_df[m].quantile(0.975), 4)
    ci_rows.append(row)

test_metrics_with_ci_df = pd.DataFrame(ci_rows).sort_values(
    "auc_point", ascending=False
).reset_index(drop=True)

test_metrics_with_ci_df.to_csv(
    SECTION_DIR / "test_metrics_with_95ci.csv",
    index=False,
    encoding="utf-8-sig"
)

print("测试集95% CI已完成。")
display(test_metrics_with_ci_df)

In [ ]:
# 4.2.1 汇总训练和测试集总表

train_part = repeated_summary_df.copy()
test_part = test_metrics_df.copy()

train_test_summary_df = train_part.merge(
    test_part,
    on="model",
    how="inner"
)

train_test_summary_df = train_test_summary_df[
    [
        "model",
        "youden_index_mean", "youden_index_std",
        "auc_mean", "auc_std",
        "auprc_mean", "auprc_std",
        "f1_mean", "f1_std",
        "sensitivity_mean", "sensitivity_std",
        "specificity_mean", "specificity_std",
        "threshold_median",
        "final_youden_median",
        "auc_test",
        "auprc_test",
        "f1_test",
        "sensitivity_test",
        "specificity_test",
        "accuracy_test",
        "balanced_accuracy_test",
        "precision_test"
    ]
].sort_values("auc_test", ascending=False).reset_index(drop=True)

train_test_summary_df.to_csv(
    SECTION_DIR / "train_test_summary_table.csv",
    index=False,
    encoding="utf-8-sig"
)

print("训练-测试总表已保存。")
display(train_test_summary_df)

In [ ]:
# 4.2.2 测试集校准分析

calibration_rows = []

plt.figure(figsize=(7, 6), dpi=300)

for model_name, y_prob in test_pred_prob_dict.items():
    y_true = y_test_sel.values

    # Brier score
    brier = brier_score_loss(y_true, y_prob)

    # 使用与 calibration curve 一致的 quantile 分箱计算 ECE
    n_bins = 10
    order = np.argsort(y_prob)
    y_prob_sorted = y_prob[order]
    y_true_sorted = y_true[order]

    bin_edges = np.linspace(0, len(y_true), n_bins + 1, dtype=int)

    ece = 0.0
    for b in range(n_bins):
        start = bin_edges[b]
        end = bin_edges[b + 1]
        if end <= start:
            continue

        prob_bin = y_prob_sorted[start:end]
        true_bin = y_true_sorted[start:end]

        bin_acc = true_bin.mean()
        bin_conf = prob_bin.mean()
        ece += (len(prob_bin) / len(y_true)) * abs(bin_acc - bin_conf)

    # calibration 截距和斜率
    # 基于原始预测概率评估，不做 post-hoc recalibration
    eps = 1e-6
    p_clip = np.clip(y_prob, eps, 1 - eps)
    logit_p = np.log(p_clip / (1 - p_clip)).reshape(-1, 1)

    lr_cal = LogisticRegression(
        fit_intercept=True,
        penalty=None,
        solver="lbfgs",
        max_iter=1000
    )
    lr_cal.fit(logit_p, y_true)

    intercept = float(lr_cal.intercept_[0])
    slope = float(lr_cal.coef_[0][0])

    # calibration 曲线
    frac_pos, mean_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=n_bins,
        strategy="quantile"
    )
    plt.plot(mean_pred, frac_pos, marker="o", linewidth=1.5, label=model_name)

    calibration_rows.append({
        "model": model_name,
        "brier": round(brier, 6),
        "ece": round(ece, 6),
        "calibration_intercept": round(intercept, 6),
        "calibration_slope": round(slope, 6)
    })

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
plt.xlabel("Predicted probability")
plt.ylabel("Observed event rate")
plt.title("Calibration Curves on Test Set")
plt.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(SECTION_DIR / "test_calibration_curves.png", dpi=300, bbox_inches="tight")
plt.show()

calibration_metrics_df = pd.DataFrame(calibration_rows).sort_values(
    "brier", ascending=True
).reset_index(drop=True)

calibration_metrics_df.to_csv(
    SECTION_DIR / "test_calibration_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

display(calibration_metrics_df)

In [ ]:
# 4.2.3 测试集 DCA 曲线

y_true = y_test_sel.values
n = len(y_true)

thresholds = np.arange(0.01, 0.25, 0.01)

dca_rows = []

# 各模型
for model_name, y_prob in test_pred_prob_dict.items():
    for pt in thresholds:
        y_pred = (y_prob >= pt).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        net_benefit = (tp / n) - (fp / n) * (pt / (1 - pt))

        dca_rows.append({
            "model": model_name,
            "threshold": pt,
            "net_benefit": net_benefit
        })

# 治疗全部
prevalence = y_true.mean()
for pt in thresholds:
    nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
    dca_rows.append({
        "model": "Treat All",
        "threshold": pt,
        "net_benefit": nb_all
    })

# 不治疗
for pt in thresholds:
    dca_rows.append({
        "model": "Treat None",
        "threshold": pt,
        "net_benefit": 0.0
    })

dca_long_df = pd.DataFrame(dca_rows)
dca_long_df.to_csv(
    SECTION_DIR / "dca_long_df.csv",
    index=False,
    encoding="utf-8-sig"
)

plt.figure(figsize=(8, 6), dpi=300)

for model_name in dca_long_df["model"].unique():
    sub = dca_long_df[dca_long_df["model"] == model_name]
    plt.plot(sub["threshold"], sub["net_benefit"], label=model_name)

plt.xlabel("Threshold probability")
plt.ylabel("Net benefit")
plt.title("Decision Curve Analysis on Test Set")
plt.legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig(SECTION_DIR / "dca_curves_test.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 4.2.4 EBM 全局主效应解释 + 1000 次 bootstrap 95% CI

ebm_main_final = final_models["EBM_main"]

shape_dir = SECTION_DIR / "ebm_main_shapes"
shape_table_dir = shape_dir / "shape_tables"

shape_dir.mkdir(parents=True, exist_ok=True)
shape_table_dir.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_N = globals().get("BOOTSTRAP_N", 1000)
N_BOOT_SHAPE = BOOTSTRAP_N
BOOT_SEED = RANDOM_STATE
STRATIFIED_BOOT = True
CI_ALPHA = 0.05
HIST_BINS = 30

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
GRAY = "#666666"

# 先统一拟合 bootstrap 模型
print(f"[Info] Fitting bootstrap models once: n_boot={N_BOOT_SHAPE}, stratified={STRATIFIED_BOOT} ...")

rng = np.random.default_rng(BOOT_SEED)
y_arr = y_train_sel.astype(int).values
idx0 = np.where(y_arr == 0)[0]
idx1 = np.where(y_arr == 1)[0]

boot_models = []
for _ in range(N_BOOT_SHAPE):
    if STRATIFIED_BOOT:
        bs0 = rng.choice(idx0, size=len(idx0), replace=True)
        bs1 = rng.choice(idx1, size=len(idx1), replace=True)
        idx = np.concatenate([bs0, bs1])
        rng.shuffle(idx)
    else:
        idx = rng.choice(np.arange(len(X_train_sel)), size=len(X_train_sel), replace=True)

    model_boot = clone(best_estimators["EBM_main"])
    model_boot.fit(
        X_train_sel.iloc[idx].reset_index(drop=True),
        y_train_sel.iloc[idx].reset_index(drop=True)
    )
    boot_models.append(model_boot)

print("[Info] Bootstrap models fitted:", len(boot_models))

shape_tables = []

feat_names_model = list(getattr(ebm_main_final, "feature_names_in_", X_train_sel.columns))
term_features_final = ebm_main_final.term_features_
term_scores_final = ebm_main_final.term_scores_
bins_final = ebm_main_final.bins_

for feat in selected_features:
    if feat not in X_train_sel.columns:
        continue

    fi = list(X_train_sel.columns).index(feat)

    # 找最终模型里这个特征对应的主效应 term
    term_idx = None
    for ti, feats in enumerate(term_features_final):
        feats = list(feats) if isinstance(feats, (list, tuple, np.ndarray)) else [feats]
        if len(feats) == 1 and int(feats[0]) == fi:
            term_idx = ti
            break

    if term_idx is None:
        print(f"[Skip] {feat}: 找不到主效应term")
        continue

    # 最终模型的 score
    s_ref = np.asarray(term_scores_final[term_idx], dtype=float).ravel()
    if s_ref.ndim > 1 and s_ref.shape[-1] == 2:
        s_ref = s_ref[..., 1].ravel()

    # cuts
    b = bins_final[fi]
    cuts = None
    if isinstance(b, np.ndarray) and b.ndim == 1:
        cuts = b.astype(float)
    elif isinstance(b, (list, tuple)):
        if len(b) == 1 and isinstance(b[0], np.ndarray) and b[0].ndim == 1:
            cuts = b[0].astype(float)
        else:
            for x in b:
                if isinstance(x, np.ndarray) and x.ndim == 1:
                    cuts = x.astype(float)
                    break

    # 判断连续变量还是分类变量
    if cuts is not None and len(cuts) > 0:
        x_raw = pd.to_numeric(X_train_sel[feat], errors="coerce").dropna().values.astype(float)
        x_min = float(np.min(x_raw))
        x_max = float(np.max(x_raw))
        if x_min == x_max:
            x_min -= 0.5
            x_max += 0.5

        edges_eff = np.concatenate(([x_min], cuts, [x_max]))
        mids = (edges_eff[:-1] + edges_eff[1:]) / 2.0

        # interpret 有时会多一个 missing bin
        if len(s_ref) == len(mids) + 1:
            s_ref = s_ref[:len(mids)]

        if len(s_ref) != len(mids):
            m = min(len(s_ref), len(mids))
            s_ref = s_ref[:m]
            mids = mids[:m]
            edges_eff = np.linspace(x_min, x_max, len(mids) + 1)

        used_real_bins = True

    else:
        mids = np.arange(len(s_ref), dtype=float)
        edges_eff = np.arange(len(s_ref) + 1, dtype=float)
        x_min, x_max = -0.5, len(s_ref) - 0.5
        used_real_bins = False

    # bootstrap shape 对齐到最终模型 mids
    boot_vals = []

    for mb in boot_models:
        feat_names_b = list(getattr(mb, "feature_names_in_", X_train_sel.columns))
        term_features_b = mb.term_features_
        term_scores_b = mb.term_scores_
        bins_b = mb.bins_

        fi_b = list(X_train_sel.columns).index(feat)

        term_idx_b = None
        for ti, feats in enumerate(term_features_b):
            feats = list(feats) if isinstance(feats, (list, tuple, np.ndarray)) else [feats]
            if len(feats) == 1 and int(feats[0]) == fi_b:
                term_idx_b = ti
                break

        if term_idx_b is None:
            continue

        s_b = np.asarray(term_scores_b[term_idx_b], dtype=float).ravel()
        if s_b.ndim > 1 and s_b.shape[-1] == 2:
            s_b = s_b[..., 1].ravel()

        b_b = bins_b[fi_b]
        cuts_b = None
        if isinstance(b_b, np.ndarray) and b_b.ndim == 1:
            cuts_b = b_b.astype(float)
        elif isinstance(b_b, (list, tuple)):
            if len(b_b) == 1 and isinstance(b_b[0], np.ndarray) and b_b[0].ndim == 1:
                cuts_b = b_b[0].astype(float)
            else:
                for x in b_b:
                    if isinstance(x, np.ndarray) and x.ndim == 1:
                        cuts_b = x.astype(float)
                        break

        if used_real_bins and cuts_b is not None and len(cuts_b) > 0:
            x_raw_b = pd.to_numeric(X_train_sel[feat], errors="coerce").dropna().values.astype(float)
            x_min_b = float(np.min(x_raw_b))
            x_max_b = float(np.max(x_raw_b))
            if x_min_b == x_max_b:
                x_min_b -= 0.5
                x_max_b += 0.5

            edges_b = np.concatenate(([x_min_b], cuts_b, [x_max_b]))
            mids_b = (edges_b[:-1] + edges_b[1:]) / 2.0

            if len(s_b) == len(mids_b) + 1:
                s_b = s_b[:len(mids_b)]

            if len(s_b) != len(mids_b):
                m = min(len(s_b), len(mids_b))
                s_b = s_b[:m]
                mids_b = mids_b[:m]
                edges_b = np.linspace(x_min_b, x_max_b, len(mids_b) + 1)

            idx_eval = np.searchsorted(edges_b[1:-1], mids, side="right")
            idx_eval = np.clip(idx_eval, 0, len(s_b) - 1)
            boot_vals.append(s_b[idx_eval])

        elif (not used_real_bins) and (cuts_b is None or len(cuts_b) == 0):
            out = np.full(len(mids), np.nan)
            m = min(len(out), len(s_b))
            out[:m] = s_b[:m]
            boot_vals.append(out)

    if len(boot_vals) > 0:
        boot_vals = np.vstack(boot_vals)
        boot_mean = np.nanmean(boot_vals, axis=0)
        ci_lo = np.nanquantile(boot_vals, CI_ALPHA / 2.0, axis=0)
        ci_hi = np.nanquantile(boot_vals, 1.0 - CI_ALPHA / 2.0, axis=0)
        n_boot_used = boot_vals.shape[0]
    else:
        boot_mean = s_ref.copy()
        ci_lo = np.full_like(s_ref, np.nan, dtype=float)
        ci_hi = np.full_like(s_ref, np.nan, dtype=float)
        n_boot_used = 0

    # 画图
    x_hist = pd.to_numeric(X_train_sel[feat], errors="coerce").dropna().values

    fig, ax1 = plt.subplots(figsize=(9.2, 5.2), dpi=300)

    if np.isfinite(ci_lo).any() and np.isfinite(ci_hi).any():
        ax1.fill_between(mids, ci_lo, ci_hi, color=BLUE, alpha=0.18, label="Bootstrap 95% CI")

    ax1.plot(mids, boot_mean, color=BLUE, linewidth=2.2, label="Bootstrap mean")
    ax1.plot(mids, s_ref, color=GRAY, linewidth=1.8, linestyle="--", label="Ref fit (full train)")
    ax1.axhline(0, color=GRAY, linewidth=1, linestyle="--")
    ax1.set_xlabel(feat)
    ax1.set_ylabel("Log-odds contribution")
    ax1.set_title(f"EBM-main stability band (standard shape): {feat} | N_BOOT={n_boot_used}")

    ax2 = ax1.twinx()
    ax2.hist(x_hist, bins=HIST_BINS, color=ORANGE, alpha=0.25)
    ax2.set_ylabel("Train distribution (count)", color=GRAY)
    ax2.tick_params(axis="y", labelcolor=GRAY)

    if used_real_bins:
        ax1.set_xlim(x_min, x_max)
    else:
        ax1.set_xlim(-0.5, len(mids) - 0.5)

    ax1.legend(loc="best", frameon=False)
    fig.tight_layout()

    out_png = shape_dir / f"EBM_main_shape_{safe_name(feat)}.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("[Saved]", out_png.resolve())

    # 导出表
    if used_real_bins:
        bin_left = edges_eff[:-1]
        bin_right = edges_eff[1:]
    else:
        bin_left = np.arange(len(mids), dtype=float)
        bin_right = np.arange(1, len(mids) + 1, dtype=float)

    table = pd.DataFrame({
        "feature": feat,
        "bin_left": bin_left,
        "bin_right": bin_right,
        "x_mid": mids,
        "score_final_log_odds": s_ref,
        "boot_mean": boot_mean,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "n_boot_used": n_boot_used
    })

    out_csv = shape_table_dir / f"shape_table_{safe_name(feat)}.csv"
    table.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[Saved]", out_csv.resolve())

    shape_tables.append(table)

# 汇总总表
if len(shape_tables) > 0:
    shape_table_df = pd.concat(shape_tables, axis=0, ignore_index=True)
    shape_table_df.to_csv(
        shape_dir / "ebm_main_shape_table_all.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("[Saved]", (shape_dir / "ebm_main_shape_table_all.csv").resolve())
    display(shape_table_df.head(20))

print("EBM_main 全局主效应解释完成。")

# 拼图
cm_dir = SECTION_DIR / "ebm_main_shapes"

cm_paths = sorted(cm_dir.glob("*.png"))

make_image_grid(
    image_paths=cm_paths,
    out_path=cm_dir / "ebm_main_shapes_grid.png",
    add_panel_labels=True
)

In [ ]:
# 4.2.5 EBM_inter 交互解释

ebm_inter = final_models["EBM_inter"]
inter_dir = SECTION_DIR / "ebm_inter_interactions"
heatmap_dir = inter_dir / "heatmaps"
inter_dir.mkdir(parents=True, exist_ok=True)
heatmap_dir.mkdir(parents=True, exist_ok=True)

TOP_N_INTER = 10   # 取前10重要的交互项
MAX_TICKS = 8
CMAP = "RdBu_r"

feat_names = list(getattr(ebm_inter, "feature_names_in_", X_train_sel.columns))
term_features = list(ebm_inter.term_features_)
term_scores = list(ebm_inter.term_scores_)
bins_all = ebm_inter.bins_
global_exp = ebm_inter.explain_global()

interaction_rows = []

# 辅助函数
def extract_cuts(bin_obj):
    cuts = None
    if isinstance(bin_obj, np.ndarray) and bin_obj.ndim == 1:
        cuts = bin_obj.astype(float)
    elif isinstance(bin_obj, (list, tuple)):
        for x in bin_obj:
            if isinstance(x, np.ndarray) and x.ndim == 1:
                cuts = x.astype(float)
                break
    return cuts

def extract_cat_labels(bin_obj, series, n_bins):
    """
    给分类变量构造标签：
    1) 若 bins_ 里有 dict，优先按 dict 的顺序取类别
    2) 否则退回到训练集实际出现的唯一值
    3) 最后与 n_bins 对齐
    """
    labels = None

    if isinstance(bin_obj, dict):
        try:
            labels = [str(k) for k, v in sorted(bin_obj.items(), key=lambda kv: kv[1])]
        except Exception:
            labels = [str(k) for k in bin_obj.keys()]
    elif isinstance(bin_obj, (list, tuple)):
        for x in bin_obj:
            if isinstance(x, dict):
                try:
                    labels = [str(k) for k, v in sorted(x.items(), key=lambda kv: kv[1])]
                except Exception:
                    labels = [str(k) for k in x.keys()]
                break

    if labels is None or len(labels) == 0:
        vals = pd.Series(series).dropna().tolist()
        uniq = []
        seen = set()
        for v in vals:
            sv = str(v)
            if sv not in seen:
                uniq.append(sv)
                seen.add(sv)
        try:
            labels = sorted(uniq, key=lambda z: float(z))
        except Exception:
            labels = uniq

    if len(labels) > n_bins:
        labels = labels[:n_bins]
    elif len(labels) < n_bins:
        labels = labels + [f"Level_{i+1}" for i in range(n_bins - len(labels))]

    return labels

def build_cont_labels(edges, n_bins):
    labels = []
    for i in range(n_bins):
        if i == 0:
            labels.append(f"≤{edges[i+1]:.2f}")
        elif i == n_bins - 1:
            labels.append(f">{edges[i]:.2f}")
        else:
            labels.append(f"({edges[i]:.2f},{edges[i+1]:.2f}]")
    return labels

def format_axis_labels(labels, n_bins):
    """
    将 global_exp 返回的轴标签统一格式化到两位小数。
    如果是纯数字字符串，就直接格式化；
    否则保留原样。
    """
    if labels is None:
        return None

    fmt_labels = []
    for x in labels:
        try:
            fmt_labels.append(f"{float(x):.2f}")
        except Exception:
            fmt_labels.append(str(x))

    if len(fmt_labels) > n_bins:
        fmt_labels = fmt_labels[:n_bins]
    elif len(fmt_labels) < n_bins:
        fmt_labels = fmt_labels + [f"Level_{i+1}" for i in range(n_bins - len(fmt_labels))]

    return fmt_labels

def extract_axis_labels_from_global(global_exp, term_idx, n1, n2):
    """
    从 explain_global() 里直接拿交互项两轴标签。
    """
    try:
        data_i = global_exp.data(term_idx)
    except Exception:
        return None, None

    x_labels = None
    y_labels = None

    # interpret 的交互项常见字段
    if isinstance(data_i, dict):
        if "left_names" in data_i and data_i["left_names"] is not None:
            x_labels = [str(x) for x in data_i["left_names"]]
        elif "names" in data_i and data_i["names"] is not None and len(data_i["names"]) == n1:
            x_labels = [str(x) for x in data_i["names"]]

        if "right_names" in data_i and data_i["right_names"] is not None:
            y_labels = [str(x) for x in data_i["right_names"]]

    x_labels = format_axis_labels(x_labels, n1)
    y_labels = format_axis_labels(y_labels, n2)

    return x_labels, y_labels


# 抽取所有二阶交互项 importance
for i, feats in enumerate(term_features):
    feats = list(feats) if isinstance(feats, (list, tuple, np.ndarray)) else [feats]
    if len(feats) != 2:
        continue

    f1 = feat_names[int(feats[0])]
    f2 = feat_names[int(feats[1])]

    Z = np.asarray(term_scores[i])
    if Z.ndim >= 1 and Z.shape[-1] == 2:
        Z = np.asarray(Z[..., 1], dtype=float)
    else:
        Z = np.asarray(Z, dtype=float)

    imp = float(np.mean(np.abs(Z)))

    interaction_rows.append({
        "term_idx": i,
        "feature_1": f1,
        "feature_2": f2,
        "importance": imp
    })

interaction_importance_df = pd.DataFrame(interaction_rows).sort_values(
    "importance", ascending=False
).reset_index(drop=True)

interaction_importance_df.to_csv(
    inter_dir / "ebm_inter_interaction_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

display(interaction_importance_df.head(10))

# 只取前 N 个最重要交互
top_interaction_df = interaction_importance_df.head(TOP_N_INTER).copy().reset_index(drop=True)

# 只对前 N 个最重要交互逐个尝试画图
manifest_rows = []

for rank, row in top_interaction_df.iterrows():
    term_idx = int(row["term_idx"])
    f1 = row["feature_1"]
    f2 = row["feature_2"]

    feats = list(term_features[term_idx])
    f1_idx = int(feats[0])
    f2_idx = int(feats[1])

    Z = np.asarray(term_scores[term_idx])
    if Z.ndim >= 1 and Z.shape[-1] == 2:
        Z = np.asarray(Z[..., 1], dtype=float)
    else:
        Z = np.asarray(Z, dtype=float)

    if Z.ndim != 2:
        manifest_rows.append({
            "rank": rank + 1,
            "feature_1": f1,
            "feature_2": f2,
            "success": False,
            "error": f"Z.shape={Z.shape}, not 2D"
        })
        continue

    n1, n2 = Z.shape


    # 优先从 global explanation 直接提取两轴标签
    x_labels, y_labels = extract_axis_labels_from_global(global_exp, term_idx, n1, n2)

    x_axis_type = "unknown"
    y_axis_type = "unknown"

    # x 轴
    if x_labels is None:
        b1 = bins_all[f1_idx]
        cuts1 = extract_cuts(b1)

        if cuts1 is not None:
            x_min = float(pd.to_numeric(X_train_sel[f1], errors="coerce").dropna().min())
            x_max = float(pd.to_numeric(X_train_sel[f1], errors="coerce").dropna().max())
            edges1 = np.concatenate(([x_min], cuts1, [x_max]))
            exp_n1 = len(edges1) - 1

            if n1 > exp_n1:
                Z = Z[:exp_n1, :]
                n1, n2 = Z.shape
            if n1 < exp_n1:
                edges1 = edges1[:n1 + 1]
                exp_n1 = len(edges1) - 1

            if n1 != exp_n1:
                manifest_rows.append({
                    "rank": rank + 1,
                    "feature_1": f1,
                    "feature_2": f2,
                    "success": False,
                    "error": f"x-axis mismatch after trim: Z.shape[0]={n1}, expected={exp_n1}"
                })
                continue

            x_labels = build_cont_labels(edges1, n1)
            x_axis_type = "continuous"
        else:
            x_labels = extract_cat_labels(b1, X_train_sel[f1], n1)
            x_axis_type = "categorical"
    else:
        x_axis_type = "from_global"

    # y 轴
    if y_labels is None:
        b2 = bins_all[f2_idx]
        cuts2 = extract_cuts(b2)

        if cuts2 is not None:
            y_min = float(pd.to_numeric(X_train_sel[f2], errors="coerce").dropna().min())
            y_max = float(pd.to_numeric(X_train_sel[f2], errors="coerce").dropna().max())
            edges2 = np.concatenate(([y_min], cuts2, [y_max]))
            exp_n2 = len(edges2) - 1

            if n2 > exp_n2:
                Z = Z[:, :exp_n2]
                n1, n2 = Z.shape
            if n2 < exp_n2:
                edges2 = edges2[:n2 + 1]
                exp_n2 = len(edges2) - 1

            if n2 != exp_n2:
                manifest_rows.append({
                    "rank": rank + 1,
                    "feature_1": f1,
                    "feature_2": f2,
                    "success": False,
                    "error": f"y-axis mismatch after trim: Z.shape[1]={n2}, expected={exp_n2}"
                })
                continue

            y_labels = build_cont_labels(edges2, n2)
            y_axis_type = "continuous"
        else:
            y_labels = extract_cat_labels(b2, X_train_sel[f2], n2)
            y_axis_type = "categorical"
    else:
        y_axis_type = "from_global"

    # 最后再性检查一次
    n1, n2 = Z.shape
    if len(x_labels) != n1 or len(y_labels) != n2:
        manifest_rows.append({
            "rank": rank + 1,
            "feature_1": f1,
            "feature_2": f2,
            "success": False,
            "error": f"label mismatch: len(x_labels)={len(x_labels)}, len(y_labels)={len(y_labels)}, Z.shape={Z.shape}"
        })
        continue

    abs_z = np.abs(Z[np.isfinite(Z)])
    v = np.quantile(abs_z, 0.99) if abs_z.size > 0 else 1e-6
    if not np.isfinite(v) or v <= 0:
        v = np.max(abs_z) if abs_z.size > 0 else 1e-6
    if not np.isfinite(v) or v <= 0:
        v = 1e-6

    out_png = heatmap_dir / f"interaction_rank{rank+1}_{safe_name(f1)}_x_{safe_name(f2)}.png"

    plt.figure(figsize=(6.5, 5.5), dpi=220)
    ax = sns.heatmap(
        Z.T,
        cmap=CMAP,
        center=0,
        vmin=-v,
        vmax=v,
        xticklabels=x_labels,
        yticklabels=y_labels,
        cbar_kws={"label": "Interaction contribution (log-odds)"}
    )

    plt.title(f"EBM interaction: {f1} × {f2}")
    plt.xlabel(f1)
    plt.ylabel(f2)

    xt = np.arange(0, n1, int(np.ceil(n1 / MAX_TICKS))) if n1 > MAX_TICKS else np.arange(n1)
    yt = np.arange(0, n2, int(np.ceil(n2 / MAX_TICKS))) if n2 > MAX_TICKS else np.arange(n2)

    ax.set_xticks(xt + 0.5)
    ax.set_yticks(yt + 0.5)
    ax.set_xticklabels(np.array(x_labels, dtype=object)[xt], rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(np.array(y_labels, dtype=object)[yt], fontsize=8)

    plt.tight_layout()
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.show()

    manifest_rows.append({
        "rank": rank + 1,
        "feature_1": f1,
        "feature_2": f2,
        "success": True,
        "axis_type": f"{x_axis_type} x {y_axis_type}",
        "figure_path": str(out_png.resolve()),
        "error": ""
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(
    inter_dir / "interaction_heatmap_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

display(manifest_df)

print(f"成功绘制热力图数量: {manifest_df['success'].sum() if len(manifest_df) > 0 else 0}")
print(f"尝试绘制的交互数量（严格前 {TOP_N_INTER} 个）: {len(top_interaction_df)}")



# 拼图
cm_dir = SECTION_DIR / "ebm_inter_interactions" / "heatmaps"

cm_paths = sorted(cm_dir.glob("*.png"))

make_image_grid(
    image_paths=cm_paths,
    out_path=cm_dir / "ebm_inter_interactions_grid.png",
    add_panel_labels=True
)

In [ ]:
# 4.2.6 典型个体病例解释（基于 EBM_main）

set_section(4)

case_dir = SECTION_DIR / "typical_cases"
case_dir.mkdir(parents=True, exist_ok=True)

ebm_main = final_models["EBM_main"]
thr_main = final_thresholds["EBM_main"]

# 测试集预测
y_prob_main = ebm_main.predict_proba(X_test_sel)[:, 1]
y_pred_main = (y_prob_main >= thr_main).astype(int)

case_df = X_test_sel.copy()
case_df["y_true"] = y_test_sel.values
case_df["y_prob"] = y_prob_main
case_df["y_pred"] = y_pred_main
case_df["abs_to_thr"] = np.abs(case_df["y_prob"] - thr_main)

# 如果前面保留了原始患者ID，就带上；没有的话就用 test index
id_test_obj = globals().get("id_test", None)

if id_test_obj is not None:
    case_df["patient_id"] = pd.Series(id_test_obj).reset_index(drop=True).astype(str)
else:
    case_df["patient_id"] = [f"test_idx_{i}" for i in range(len(case_df))]

# 选 3 类典型病例
# -----------------------------
# 高风险真阳性
high_tp_df = case_df[(case_df["y_true"] == 1) & (case_df["y_pred"] == 1)].sort_values(
    "y_prob", ascending=False
)

# 最接近阈值的病例
near_thr_df = case_df.sort_values("abs_to_thr", ascending=True)

# 低风险真阴性
low_tn_df = case_df[(case_df["y_true"] == 0) & (case_df["y_pred"] == 0)].sort_values(
    "y_prob", ascending=True
)

picked_cases = {}

if len(high_tp_df) > 0:
    picked_cases["high_risk_true_positive"] = high_tp_df.index[0]

if len(near_thr_df) > 0:
    picked_cases["near_threshold_case"] = near_thr_df.index[0]

if len(low_tn_df) > 0:
    picked_cases["low_risk_true_negative"] = low_tn_df.index[0]

picked_case_rows = []


# 循环输出每个病例
for case_name, idx in picked_cases.items():
    row_x = X_test_sel.iloc[[idx]]
    row_info = case_df.loc[idx]

    patient_id = row_info["patient_id"]

    picked_case_rows.append({
        "case_name": case_name,
        "patient_id": str(patient_id),
        "test_index": int(idx),
        "y_true": int(row_info["y_true"]),
        "y_pred": int(row_info["y_pred"]),
        "y_prob": round(float(row_info["y_prob"]), 6),
        "threshold": round(float(thr_main), 6),
        "abs_to_threshold": round(float(row_info["abs_to_thr"]), 6)
    })

    # local explanation
    local_exp = ebm_main.explain_local(row_x, y=[row_info["y_true"]])
    data0 = local_exp.data(0)

    feat_names = list(data0["names"])
    feat_scores = np.array(data0["scores"], dtype=float)

    local_df = pd.DataFrame({
        "feature": feat_names,
        "contribution": feat_scores
    })

    # 加绝对值，便于看主导因素
    local_df["abs_contribution"] = np.abs(local_df["contribution"])
    local_df = local_df.sort_values("contribution", ascending=True).reset_index(drop=True)

    # 导出 csv
    local_df.to_csv(
        case_dir / f"{case_name}_local_explanation.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 颜色：正向风险提升 / 负向风险降低
    bar_colors = ["#d62728" if x > 0 else "#1f77b4" for x in local_df["contribution"]]

    # 画图
    fig_h = max(4.5, len(local_df) * 0.42)
    plt.figure(figsize=(7.5, fig_h), dpi=300)

    plt.barh(
        local_df["feature"],
        local_df["contribution"],
        color=bar_colors
    )
    plt.axvline(0, color="gray", linestyle="--", linewidth=1)

    plt.xlabel("Contribution to log-odds")
    plt.ylabel("Feature")
    plt.title(
        f"{case_name}\n"
        f"Patient={patient_id} | test_idx={idx} | "
        f"pred_prob={row_info['y_prob']:.3f} | true={int(row_info['y_true'])} | pred={int(row_info['y_pred'])}"
    )
    plt.tight_layout()

    out_png = case_dir / f"{case_name}_local_explanation.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

    print("[Saved]", out_png.resolve())
    print("[Saved]", (case_dir / f"{case_name}_local_explanation.csv").resolve())


# 汇总病例表
typical_cases_df = pd.DataFrame(picked_case_rows)
typical_cases_df.to_csv(
    case_dir / "typical_cases_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("[Saved]", (case_dir / "typical_cases_summary.csv").resolve())
display(typical_cases_df)

In [ ]:
# 5.0 固定第五章输出目录

set_section(5)

scorecard_dir = SECTION_DIR / "scorecard"
scorecard_dir.mkdir(parents=True, exist_ok=True)
print("当前输出目录:", scorecard_dir.resolve())

In [ ]:
# 5.1 参数、少量工具函数

ebm_main = final_models["EBM_main"]
thr_main = final_thresholds["EBM_main"]

POINTS_FACTOR = 20
WARNING_RECALL_SHARE = 0.30
ETA_CUT = float(np.log(thr_main / (1 - thr_main)))   # 用最终概率阈值映射到 logit 空间

global_exp = ebm_main.explain_global()
term_names = list(global_exp.data()["names"])
model_feat_names = list(getattr(ebm_main, "feature_names_in_", X_train_sel.columns))
bins_all = ebm_main.bins_


def extract_cuts(bin_obj):
    if isinstance(bin_obj, np.ndarray) and bin_obj.ndim == 1:
        return bin_obj.astype(float)
    if isinstance(bin_obj, (list, tuple)):
        for x in bin_obj:
            if isinstance(x, np.ndarray) and x.ndim == 1:
                return x.astype(float)
    return None


def extract_intercept(model):
    itc = getattr(model, "intercept_", 0.0)
    if isinstance(itc, (list, tuple, np.ndarray)):
        arr = np.ravel(itc)
        return float(arr[0]) if len(arr) > 0 else 0.0
    return float(itc)


def shape_complexity(scores):
    scores = np.asarray(scores, dtype=float)
    if len(scores) <= 1:
        return 0, 0, 0, 0.0

    d1 = np.diff(scores)
    ad1 = np.abs(d1)
    zero_crossings = int(np.sum(scores[:-1] * scores[1:] <= 0))

    if np.any(ad1 > 0):
        jump_thr = np.quantile(ad1[ad1 > 0], 0.75)
        large_jumps = int(np.sum(ad1 >= jump_thr)) if jump_thr > 0 else 0
    else:
        large_jumps = 0

    s1 = np.sign(d1)
    inflections = int(np.sum(s1[:-1] * s1[1:] < 0)) if len(s1) >= 2 else 0

    complexity_score = float(zero_crossings + large_jumps + inflections)
    return zero_crossings, large_jumps, inflections, complexity_score


def build_interval_labels(cuts):
    if len(cuts) == 0:
        return ["All"]
    labels = []
    for i in range(len(cuts) + 1):
        if i == 0:
            labels.append(f"≤{cuts[i]:.3g}")
        elif i == len(cuts):
            labels.append(f">{cuts[i-1]:.3g}")
        else:
            labels.append(f"({cuts[i-1]:.3g}, {cuts[i]:.3g}]")
    return labels


def greedy_select_boundaries(raw_scores, raw_counts, raw_cuts, target_bins, min_bin_size):
    raw_scores = np.asarray(raw_scores, dtype=float)
    raw_counts = np.asarray(raw_counts, dtype=float)
    raw_cuts = np.asarray(raw_cuts, dtype=float)

    k = len(raw_scores)
    if k <= 1:
        return np.array([], dtype=float), [raw_scores], [raw_counts]

    target_bins = int(max(2, min(target_bins, k)))
    max_feasible_bins = max(2, int(np.floor(raw_counts.sum() / min_bin_size))) if min_bin_size > 0 else k
    target_bins = int(min(target_bins, max_feasible_bins, k))

    if k <= target_bins:
        return raw_cuts.copy(), [[s] for s in raw_scores], [[c] for c in raw_counts]

    d1 = np.diff(raw_scores)
    ad1 = np.abs(d1)
    s1 = np.sign(d1)

    boundary_strength = np.zeros(k - 1, dtype=float)
    for j in range(k - 1):
        strength = ad1[j] * np.sqrt(max(min(raw_counts[j], raw_counts[j + 1]), 1.0))
        if raw_scores[j] * raw_scores[j + 1] <= 0:
            strength *= 1.2
        if 0 < j < len(s1) - 1 and s1[j - 1] * s1[j] < 0:
            strength *= 1.2
        boundary_strength[j] = strength

    order = list(np.argsort(-boundary_strength))
    selected = []

    def segments_ok(boundaries_idx):
        start = 0
        for b in sorted(boundaries_idx) + [k]:
            if raw_counts[start:b].sum() < min_bin_size:
                return False
            start = b
        return True

    for j in order:
        if len(selected) >= target_bins - 1:
            break
        cand = sorted(selected + [j + 1])
        if segments_ok(cand):
            selected = cand

    if len(selected) < target_bins - 1:
        cum_n = np.cumsum(raw_counts)
        total_n = raw_counts.sum()
        for q in np.linspace(0, total_n, target_bins + 1)[1:-1]:
            b = int(np.searchsorted(cum_n, q, side="left")) + 1
            if 1 <= b <= k - 1 and b not in selected:
                cand = sorted(selected + [b])
                if segments_ok(cand):
                    selected = cand
            if len(selected) >= target_bins - 1:
                break

    selected = sorted(selected)

    merged_score_groups = []
    merged_count_groups = []
    start = 0
    for b in selected + [k]:
        merged_score_groups.append(list(raw_scores[start:b]))
        merged_count_groups.append(list(raw_counts[start:b]))
        start = b

    selected_cuts = raw_cuts[np.array(selected) - 1] if len(selected) > 0 else np.array([], dtype=float)
    return selected_cuts, merged_score_groups, merged_count_groups


def apply_scorecard(X_df, scorecard_spec, points_factor, intercept_value, eta_cut):
    n = len(X_df)
    total_logit = np.full(n, intercept_value, dtype=float)

    for feat in selected_features:
        spec = scorecard_spec[feat]

        if spec["feature_type"] == "continuous":
            cuts = np.array(spec["cuts"], dtype=float)
            scores = np.array(spec["scores"], dtype=float)
            x = pd.to_numeric(X_df[feat], errors="coerce").values.astype(float)
            idx = np.digitize(x, bins=cuts, right=True)
            idx = np.clip(idx, 0, len(scores) - 1)
            contrib = scores[idx]
            nan_mask = np.isnan(x)
            if np.any(nan_mask):
                contrib[nan_mask] = float(spec["default_score"])
        else:
            mp = {str(spec["levels"][i]): float(spec["scores"][i]) for i in range(len(spec["levels"]))}
            contrib = np.array(
                [mp.get(str(v), float(spec["default_score"])) for v in X_df[feat].values],
                dtype=float
            )

        total_logit += contrib

    score_raw = points_factor * (total_logit - eta_cut)
    score_round = np.round(score_raw).astype(int)
    prob = 1 / (1 + np.exp(-total_logit))
    return total_logit, prob, score_raw, score_round


def explain_one_case(x_one_df, scorecard_spec, points_factor):
    rows = []

    for feat in selected_features:
        spec = scorecard_spec[feat]
        val = x_one_df.iloc[0][feat]

        if spec["feature_type"] == "continuous":
            cuts = np.array(spec["cuts"], dtype=float)
            scores = np.array(spec["scores"], dtype=float)
            x = pd.to_numeric(pd.Series([val]), errors="coerce").iloc[0]

            if pd.isna(x):
                chosen_score = float(spec["default_score"])
                chosen_label = "Missing"
            else:
                idx = int(np.digitize([x], bins=cuts, right=True)[0])
                idx = int(np.clip(idx, 0, len(scores) - 1))
                chosen_score = float(scores[idx])
                chosen_label = spec["labels"][idx]
        else:
            mp = {str(spec["levels"][i]): float(spec["scores"][i]) for i in range(len(spec["levels"]))}
            chosen_score = float(mp.get(str(val), float(spec["default_score"])))
            chosen_label = str(val) if str(val) in [str(v) for v in spec["levels"]] else "Other/Missing"

        rows.append({
            "feature": feat,
            "raw_value": val,
            "bin_or_level": chosen_label,
            "ebm_score": chosen_score,
            "points": float(points_factor * chosen_score)
        })

    out_df = pd.DataFrame(rows)
    out_df["abs_points"] = out_df["points"].abs()
    return out_df


def risk_group_from_score(score, positive_cut, warn_cut):
    if score >= positive_cut:
        return "positive_by_scorecard"
    if score >= warn_cut:
        return "warning_negative_followup"
    return "low_risk_negative"


intercept_value = extract_intercept(ebm_main)
print(f"EBM_main 最终阈值（概率）: {thr_main:.6f}")
print(f"中心化边界 logit: {ETA_CUT:.6f}")

In [ ]:
# 5.2 简化分箱：基于主效应生成评分卡规则

term_importance_values = ebm_main.term_importances()
term_importance_map = {}
for i, nm in enumerate(term_names):
    if i < len(term_importance_values):
        term_importance_map[nm] = float(term_importance_values[i])

N_train = len(X_train_sel)
MIN_BIN_SIZE = max(int(np.ceil(0.03 * N_train)), 100)

raw_feature_store = {}
continuous_feature_stats = []
scorecard_rows = []
bin_plan_rows = []
scorecard_spec = {}

for feat in selected_features:
    if feat not in term_names:
        continue

    term_idx = term_names.index(feat)
    feat_idx = model_feat_names.index(feat)
    data_i = global_exp.data(term_idx)

    raw_names = list(data_i["names"])
    raw_scores = np.array(data_i["scores"], dtype=float)
    cuts = extract_cuts(bins_all[feat_idx])

    # 分类变量：不再额外简化，直接保留 level -> score
    if cuts is None:
        m = min(len(raw_names), len(raw_scores))
        levels = [str(x) for x in raw_names[:m]]
        scores = raw_scores[:m]
        default_score = float(np.average(scores)) if len(scores) > 0 else 0.0

        scorecard_spec[feat] = {
            "feature_type": "categorical",
            "levels": levels,
            "scores": [float(x) for x in scores],
            "labels": levels,
            "default_score": default_score
        }

        for lv, sc in zip(levels, scores):
            scorecard_rows.append({
                "feature": feat,
                "bin_or_level": lv,
                "simplified_ebm_score": round(float(sc), 6),
                "points": round(POINTS_FACTOR * float(sc), 2)
            })

        bin_plan_rows.append({
            "feature": feat,
            "feature_type": "categorical",
            "target_bins": int(len(levels)),
            "actual_bins": int(len(levels)),
            "min_bin_size_rule": MIN_BIN_SIZE
        })
        continue

    # 连续变量：按重要性+复杂度决定目标分箱数，再做简化
    cuts = np.asarray(cuts, dtype=float)
    k = min(len(raw_scores), len(cuts) + 1)
    raw_scores = raw_scores[:k]
    raw_cuts = cuts[:k - 1]

    x_feat = pd.to_numeric(X_train_sel[feat], errors="coerce").values.astype(float)
    raw_bin_idx = np.digitize(x_feat, bins=raw_cuts, right=True)
    raw_counts = np.bincount(raw_bin_idx[~np.isnan(x_feat)], minlength=k).astype(float)

    zero_crossings, large_jumps, inflections, complexity_score = shape_complexity(raw_scores)
    importance = float(term_importance_map.get(feat, np.mean(np.abs(raw_scores))))

    raw_feature_store[feat] = {
        "raw_scores": raw_scores,
        "raw_cuts": raw_cuts,
        "raw_counts": raw_counts,
        "importance": importance,
        "complexity_score": complexity_score,
        "zero_crossings": zero_crossings,
        "large_jumps": large_jumps,
        "inflections": inflections
    }

    continuous_feature_stats.append({
        "feature": feat,
        "importance": importance,
        "complexity_score": complexity_score
    })

continuous_stat_df = pd.DataFrame(continuous_feature_stats)

if len(continuous_stat_df) > 0:
    imp_q1 = float(continuous_stat_df["importance"].quantile(1 / 3))
    imp_q2 = float(continuous_stat_df["importance"].quantile(2 / 3))
    cpx_q1 = float(continuous_stat_df["complexity_score"].quantile(1 / 3))
    cpx_q2 = float(continuous_stat_df["complexity_score"].quantile(2 / 3))
else:
    imp_q1 = imp_q2 = cpx_q1 = cpx_q2 = 0.0

for feat in continuous_stat_df["feature"].tolist():
    r = continuous_stat_df.loc[continuous_stat_df["feature"] == feat].iloc[0]
    store = raw_feature_store[feat]

    if r["importance"] <= imp_q1:
        importance_level = 0
    elif r["importance"] <= imp_q2:
        importance_level = 1
    else:
        importance_level = 2

    if r["complexity_score"] <= cpx_q1:
        complexity_level = 0
    elif r["complexity_score"] <= cpx_q2:
        complexity_level = 1
    else:
        complexity_level = 2

    target_bins = int(np.clip(3 + importance_level + complexity_level, 3, 8))

    selected_cuts, merged_score_groups, merged_count_groups = greedy_select_boundaries(
        raw_scores=store["raw_scores"],
        raw_counts=store["raw_counts"],
        raw_cuts=store["raw_cuts"],
        target_bins=target_bins,
        min_bin_size=MIN_BIN_SIZE
    )

    merged_scores = []
    for sg, cg in zip(merged_score_groups, merged_count_groups):
        sg_arr = np.asarray(sg, dtype=float)
        cg_arr = np.asarray(cg, dtype=float)
        merged_scores.append(float(np.average(sg_arr, weights=np.maximum(cg_arr, 1))))

    labels = build_interval_labels(selected_cuts)
    default_score = float(np.average(merged_scores)) if len(merged_scores) > 0 else 0.0

    scorecard_spec[feat] = {
        "feature_type": "continuous",
        "cuts": [float(x) for x in selected_cuts],
        "scores": [float(x) for x in merged_scores],
        "labels": labels,
        "default_score": default_score
    }

    for lb, sc in zip(labels, merged_scores):
        scorecard_rows.append({
            "feature": feat,
            "bin_or_level": lb,
            "simplified_ebm_score": round(float(sc), 6),
            "points": round(POINTS_FACTOR * float(sc), 2)
        })

    bin_plan_rows.append({
        "feature": feat,
        "feature_type": "continuous",
        "importance": round(float(store["importance"]), 6),
        "complexity_score": round(float(store["complexity_score"]), 6),
        "zero_crossings": int(store["zero_crossings"]),
        "large_jumps": int(store["large_jumps"]),
        "inflections": int(store["inflections"]),
        "target_bins": int(target_bins),
        "actual_bins": int(len(merged_scores)),
        "min_bin_size_rule": int(MIN_BIN_SIZE)
    })

scorecard_bin_plan_df = pd.DataFrame(bin_plan_rows)
scorecard_table_df = pd.DataFrame(scorecard_rows)

display(scorecard_bin_plan_df)
display(scorecard_table_df.head(30))

In [ ]:
# 5.3 保存评分卡主体：表格、规则、元信息

scorecard_table_df.to_csv(
    scorecard_dir / "ebm_main_scorecard_table.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(scorecard_dir / "scorecard_bin_spec.json", "w", encoding="utf-8") as f:
    json.dump(scorecard_spec, f, ensure_ascii=False, indent=2)

scorecard_meta = {
    "points_factor": POINTS_FACTOR,
    "intercept_value": float(intercept_value),
    "final_prob_threshold": float(thr_main),
    "eta_cut": float(ETA_CUT),
    "warning_recall_share_target": float(WARNING_RECALL_SHARE),
    "min_bin_size_rule": int(MIN_BIN_SIZE),
    "bin_allocation_rule": "target_bins = clip(3 + importance_level + complexity_level, 3, 8)",
    "score_formula": "Score = points_factor * (simplified_logit - eta_cut)",
    "positive_rule": "score >= 0",
    "warning_rule": "Among score < 0, choose a warning threshold on train set to rescue about 30% of missed positives"
}

with open(scorecard_dir / "scorecard_meta.json", "w", encoding="utf-8") as f:
    json.dump(scorecard_meta, f, ensure_ascii=False, indent=2)

print("[Saved]", (scorecard_dir / "ebm_main_scorecard_table.csv").resolve())
print("[Saved]", (scorecard_dir / "scorecard_bin_spec.json").resolve())
print("[Saved]", (scorecard_dir / "scorecard_meta.json").resolve())

In [ ]:
# 5.4 训练集定主边界和预警边界（使用 score_raw 与整数边界比较）

train_logit_score, train_prob_score, train_score_raw, train_score_round = apply_scorecard(
    X_df=X_train_sel,
    scorecard_spec=scorecard_spec,
    points_factor=POINTS_FACTOR,
    intercept_value=intercept_value,
    eta_cut=ETA_CUT
)

score_train_df = pd.DataFrame({
    "y_true": y_train_sel.values,
    "prob": train_prob_score,
    "score_raw": train_score_raw,
    "score_round": train_score_round
})

# 主边界固定为 0 分（整数边界，但比较对象是 score_raw）
score_cut_positive_round = 0

train_pred_main = (score_train_df["score_raw"] >= score_cut_positive_round).astype(int)
tp0 = int(((train_pred_main == 1) & (score_train_df["y_true"] == 1)).sum())
pos_n = int((score_train_df["y_true"] == 1).sum())
main_sensitivity_train = tp0 / pos_n if pos_n > 0 else 0.0

# 在阴性里再额外预警约 30% 的漏检阳性
target_sensitivity_train = main_sensitivity_train + WARNING_RECALL_SHARE * (1 - main_sensitivity_train)

# 预警边界仍限定为负整数，但比较对象改为 score_raw
min_score_raw = float(np.floor(score_train_df["score_raw"].min()))
neg_candidates = list(range(int(min_score_raw), 0))   # 例如 [-12, -11, ..., -1]

score_cut_warn_round = -1
actual_target_sensitivity_train = main_sensitivity_train

if len(neg_candidates) > 0 and pos_n > 0:
    best_gap = np.inf
    best_threshold = neg_candidates[0]
    best_cum_sens = main_sensitivity_train

    for t in neg_candidates:
        cum_sens = float(((score_train_df["score_raw"] >= t) & (score_train_df["y_true"] == 1)).sum()) / pos_n
        gap = abs(cum_sens - target_sensitivity_train)

        if (gap < best_gap) or (np.isclose(gap, best_gap) and t < best_threshold):
            best_gap = gap
            best_threshold = int(t)
            best_cum_sens = cum_sens

    score_cut_warn_round = int(best_threshold)
    actual_target_sensitivity_train = float(best_cum_sens)

score_train_df["risk_group"] = score_train_df["score_raw"].apply(
    lambda s: risk_group_from_score(s, score_cut_positive_round, score_cut_warn_round)
)

warning_band_train_df = score_train_df[
    (score_train_df["score_raw"] >= score_cut_warn_round) &
    (score_train_df["score_raw"] < score_cut_positive_round)
].copy()

missed_positive_n_main = int(((score_train_df["score_raw"] < score_cut_positive_round) & (score_train_df["y_true"] == 1)).sum())
rescued_positive_n = int((warning_band_train_df["y_true"] == 1).sum())
rescued_share_of_missed = rescued_positive_n / missed_positive_n_main if missed_positive_n_main > 0 else 0.0

warning_stats_train_df = pd.DataFrame([{
    "main_sensitivity_train": round(float(main_sensitivity_train), 6),
    "target_sensitivity_train": round(float(target_sensitivity_train), 6),
    "actual_target_sensitivity_train": round(float(actual_target_sensitivity_train), 6),
    "missed_positive_n_main": int(missed_positive_n_main),
    "rescued_positive_n": int(rescued_positive_n),
    "rescued_share_of_missed": round(float(rescued_share_of_missed), 6),
    "warning_threshold_score": int(score_cut_warn_round),
    "warning_band_n": int(len(warning_band_train_df)),
    "warning_band_positive_rate": round(float(warning_band_train_df["y_true"].mean()), 6) if len(warning_band_train_df) > 0 else np.nan
}])

warning_stats_train_df.to_csv(
    scorecard_dir / "score_warning_stats_train.csv",
    index=False,
    encoding="utf-8-sig"
)

scorecard_meta["score_cut_positive_round"] = int(score_cut_positive_round)
scorecard_meta["score_cut_warn_round"] = int(score_cut_warn_round)
scorecard_meta["main_sensitivity_train"] = float(main_sensitivity_train)
scorecard_meta["target_sensitivity_train"] = float(target_sensitivity_train)
scorecard_meta["actual_target_sensitivity_train"] = float(actual_target_sensitivity_train)
scorecard_meta["rescued_share_of_missed_train"] = float(rescued_share_of_missed)

with open(scorecard_dir / "scorecard_meta.json", "w", encoding="utf-8") as f:
    json.dump(scorecard_meta, f, ensure_ascii=False, indent=2)

print(f"主边界：score_raw >= {score_cut_positive_round}")
print(f"预警边界：score_raw >= {score_cut_warn_round} 且 score_raw < {score_cut_positive_round}")
print(f"训练集主边界敏感度：{main_sensitivity_train:.4f}")
print(f"训练集目标累计敏感度：{target_sensitivity_train:.4f}")
print(f"训练集实际累计敏感度：{actual_target_sensitivity_train:.4f}")

display(warning_stats_train_df)

In [ ]:
# 5.5 测试集验证：评分卡 vs 原始 EBM-main（使用 score_raw 与整数边界比较）

with open(scorecard_dir / "scorecard_meta.json", "r", encoding="utf-8") as f:
    scorecard_meta = json.load(f)

with open(scorecard_dir / "scorecard_bin_spec.json", "r", encoding="utf-8") as f:
    scorecard_spec = json.load(f)

score_cut_positive_round = int(scorecard_meta["score_cut_positive_round"])
score_cut_warn_round = int(scorecard_meta["score_cut_warn_round"])
intercept_value = float(scorecard_meta["intercept_value"])
ETA_CUT = float(scorecard_meta["eta_cut"])

# 原始 EBM-main
test_prob_ebm = ebm_main.predict_proba(X_test_sel)[:, 1]
test_metric_ebm = compute_metrics(
    y_true=y_test_sel.values,
    y_prob=test_prob_ebm,
    threshold=thr_main
)

# 简化评分卡
test_logit_score, test_prob_score, test_score_raw, test_score_round = apply_scorecard(
    X_df=X_test_sel,
    scorecard_spec=scorecard_spec,
    points_factor=POINTS_FACTOR,
    intercept_value=intercept_value,
    eta_cut=ETA_CUT
)

# 主边界和预警边界：都用 score_raw 与整数阈值比较
test_pred_main = (test_score_raw >= score_cut_positive_round).astype(int)
test_pred_warn = (test_score_raw >= score_cut_warn_round).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_sel.values, test_pred_main).ravel()
sensitivity_main = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity_main = tn / (tn + fp) if (tn + fp) > 0 else 0.0

tn_w, fp_w, fn_w, tp_w = confusion_matrix(y_test_sel.values, test_pred_warn).ravel()
sensitivity_warn = tp_w / (tp_w + fn_w) if (tp_w + fn_w) > 0 else 0.0
specificity_warn = tn_w / (tn_w + fp_w) if (tn_w + fp_w) > 0 else 0.0

test_metric_scorecard = {
    "auc": roc_auc_score(y_test_sel.values, test_prob_score),
    "auprc": average_precision_score(y_test_sel.values, test_prob_score),
    "accuracy": accuracy_score(y_test_sel.values, test_pred_main),
    "balanced_accuracy": balanced_accuracy_score(y_test_sel.values, test_pred_main),
    "precision": precision_score(y_test_sel.values, test_pred_main, zero_division=0),
    "sensitivity": sensitivity_main,
    "specificity": specificity_main,
    "f1": f1_score(y_test_sel.values, test_pred_main, zero_division=0)
}

scorecard_compare_df = pd.DataFrame([
    {
        "model": "EBM_main_original",
        "threshold_prob": round(float(thr_main), 6),
        "threshold_score": np.nan,
        "auc": round(test_metric_ebm["auc"], 6),
        "auprc": round(test_metric_ebm["auprc"], 6),
        "accuracy": round(test_metric_ebm["accuracy"], 6),
        "balanced_accuracy": round(test_metric_ebm["balanced_accuracy"], 6),
        "precision": round(test_metric_ebm["precision"], 6),
        "sensitivity": round(test_metric_ebm["sensitivity"], 6),
        "specificity": round(test_metric_ebm["specificity"], 6),
        "f1": round(test_metric_ebm["f1"], 6),
        "brier": round(brier_score_loss(y_test_sel.values, test_prob_ebm), 6)
    },
    {
        "model": "EBM_main_centered_scorecard",
        "threshold_prob": round(float(thr_main), 6),
        "threshold_score": int(score_cut_positive_round),
        "auc": round(test_metric_scorecard["auc"], 6),
        "auprc": round(test_metric_scorecard["auprc"], 6),
        "accuracy": round(test_metric_scorecard["accuracy"], 6),
        "balanced_accuracy": round(test_metric_scorecard["balanced_accuracy"], 6),
        "precision": round(test_metric_scorecard["precision"], 6),
        "sensitivity": round(test_metric_scorecard["sensitivity"], 6),
        "specificity": round(test_metric_scorecard["specificity"], 6),
        "f1": round(test_metric_scorecard["f1"], 6),
        "brier": round(brier_score_loss(y_test_sel.values, test_prob_score), 6)
    }
])

scorecard_compare_df.to_csv(
    scorecard_dir / "scorecard_vs_ebm_main_test_compare.csv",
    index=False,
    encoding="utf-8-sig"
)

scorecard_test_df = pd.DataFrame({
    "row_id": X_test_sel.index.values,
    "y_true": y_test_sel.values,
    "scorecard_prob": test_prob_score,
    "score_raw": test_score_raw,
    "score_round": test_score_round
})

scorecard_test_df["risk_group"] = scorecard_test_df["score_raw"].apply(
    lambda s: risk_group_from_score(s, score_cut_positive_round, score_cut_warn_round)
)

warning_band_test_df = scorecard_test_df[
    (scorecard_test_df["score_raw"] >= score_cut_warn_round) &
    (scorecard_test_df["score_raw"] < score_cut_positive_round)
].copy()

missed_positive_n_main_test = int(((scorecard_test_df["score_raw"] < score_cut_positive_round) & (scorecard_test_df["y_true"] == 1)).sum())
rescued_positive_n_test = int((warning_band_test_df["y_true"] == 1).sum())
rescued_share_of_missed_test = rescued_positive_n_test / missed_positive_n_main_test if missed_positive_n_main_test > 0 else 0.0

warning_stats_test_df = pd.DataFrame([{
    "main_sensitivity_test": round(float(sensitivity_main), 6),
    "cumulative_sensitivity_test": round(float(sensitivity_warn), 6),
    "specificity_at_warning_threshold_test": round(float(specificity_warn), 6),
    "missed_positive_n_main_test": int(missed_positive_n_main_test),
    "rescued_positive_n_test": int(rescued_positive_n_test),
    "rescued_share_of_missed_test": round(float(rescued_share_of_missed_test), 6),
    "warning_threshold_score": int(score_cut_warn_round),
    "warning_band_n_test": int(len(warning_band_test_df)),
    "warning_band_positive_rate_test": round(float(warning_band_test_df["y_true"].mean()), 6) if len(warning_band_test_df) > 0 else np.nan
}])

risk_group_summary_df = (
    scorecard_test_df
    .groupby("risk_group", as_index=False)
    .agg(
        n=("y_true", "size"),
        n_positive=("y_true", "sum"),
        observed_rate=("y_true", "mean"),
        mean_score_raw=("score_raw", "mean"),
        mean_score_round=("score_round", "mean"),
        mean_prob=("scorecard_prob", "mean")
    )
)

risk_group_summary_df["observed_rate"] = risk_group_summary_df["observed_rate"].round(6)
risk_group_summary_df["mean_score_raw"] = risk_group_summary_df["mean_score_raw"].round(3)
risk_group_summary_df["mean_score_round"] = risk_group_summary_df["mean_score_round"].round(3)
risk_group_summary_df["mean_prob"] = risk_group_summary_df["mean_prob"].round(6)

warning_stats_test_df.to_csv(
    scorecard_dir / "score_warning_stats_test.csv",
    index=False,
    encoding="utf-8-sig"
)
risk_group_summary_df.to_csv(
    scorecard_dir / "score_risk_group_summary_test.csv",
    index=False,
    encoding="utf-8-sig"
)

scorecard_test_df.to_csv(
    scorecard_dir / "scorecard_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("[Saved]", (scorecard_dir / "scorecard_vs_ebm_main_test_compare.csv").resolve())
print("[Saved]", (scorecard_dir / "score_warning_stats_test.csv").resolve())
print("[Saved]", (scorecard_dir / "score_risk_group_summary_test.csv").resolve())
print("[Saved]", (scorecard_dir / "scorecard_test_predictions.csv").resolve())

print(f"主边界：score_raw >= {score_cut_positive_round}")
print(f"预警边界：score_raw >= {score_cut_warn_round} 且 score_raw < {score_cut_positive_round}")
print(f"测试集主边界敏感度：{sensitivity_main:.4f}")
print(f"测试集累计敏感度：{sensitivity_warn:.4f}")

display(scorecard_compare_df)
display(warning_stats_test_df)
display(risk_group_summary_df)

In [ ]:
# 5.6 三层代表病例示例（保留原始小数评分，与整数边界比较）

if "scorecard_test_df" not in globals():
    scorecard_test_df = pd.read_csv(scorecard_dir / "scorecard_test_predictions.csv")

with open(scorecard_dir / "scorecard_meta.json", "r", encoding="utf-8") as f:
    scorecard_meta = json.load(f)

with open(scorecard_dir / "scorecard_bin_spec.json", "r", encoding="utf-8") as f:
    scorecard_spec = json.load(f)

score_cut_positive_round = int(scorecard_meta["score_cut_positive_round"])
score_cut_warn_round = int(scorecard_meta["score_cut_warn_round"])

example_rows = []
example_detail_frames = []

for group_name in [
    "low_risk_negative",
    "warning_negative_followup",
    "positive_by_scorecard"
]:
    group_df = scorecard_test_df[scorecard_test_df["risk_group"] == group_name].copy()
    if len(group_df) == 0:
        continue

    # 按 score_raw 选代表病例
    group_df["dist_to_median"] = (group_df["score_raw"] - group_df["score_raw"].median()).abs()
    picked = group_df.sort_values(
        ["dist_to_median", "score_raw"],
        ascending=[True, False]
    ).iloc[0]

    row_id = picked["row_id"]
    x_one = X_test_sel.loc[[row_id]].copy()

    detail_df = explain_one_case(
        x_one_df=x_one,
        scorecard_spec=scorecard_spec,
        points_factor=POINTS_FACTOR
    ).sort_values("points", ascending=False)

    detail_df["risk_group"] = group_name
    detail_df["row_id"] = row_id
    detail_df["score_raw_total"] = float(picked["score_raw"])
    example_detail_frames.append(detail_df)

    if group_name == "low_risk_negative":
        score_rule = f"score < {score_cut_warn_round}"
    elif group_name == "warning_negative_followup":
        score_rule = f"{score_cut_warn_round} <= score < {score_cut_positive_round}"
    else:
        score_rule = f"score >= {score_cut_positive_round}"

    example_rows.append({
        "risk_group": group_name,
        "row_id": row_id,
        "y_true": int(picked["y_true"]),
        "score_raw": round(float(picked["score_raw"]), 4),
        "score_round": int(picked["score_round"]),
        "scorecard_prob": round(float(picked["scorecard_prob"]), 6),
        "score_rule": score_rule
    })

scorecard_example_df = pd.DataFrame(example_rows)
scorecard_example_detail_df = (
    pd.concat(example_detail_frames, axis=0, ignore_index=True)
    if len(example_detail_frames) > 0 else pd.DataFrame()
)

scorecard_example_df.to_csv(
    scorecard_dir / "scorecard_example_cases.csv",
    index=False,
    encoding="utf-8-sig"
)
scorecard_example_detail_df.to_csv(
    scorecard_dir / "scorecard_example_case_details.csv",
    index=False,
    encoding="utf-8-sig"
)

print("[Saved]", (scorecard_dir / "scorecard_example_cases.csv").resolve())
print("[Saved]", (scorecard_dir / "scorecard_example_case_details.csv").resolve())

display(scorecard_example_df)
display(scorecard_example_detail_df.head(30))

In [ ]:
# 5.7 三层代表病例解释图（显示小数评分，与整数边界对比）

def plot_one_example_case(ax, detail_df, title, top_n=10):
    plot_df = (
        detail_df[["feature", "bin_or_level", "points"]]
        .copy()
        .assign(abs_points=lambda x: x["points"].abs())
        .sort_values(["abs_points", "points"], ascending=[False, True])
        .head(top_n)
        .sort_values("points", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(plot_df))
    vals = plot_df["points"].to_numpy()

    labels = [
        f"{feat} = {grp}"
        for feat, grp in zip(plot_df["feature"], plot_df["bin_or_level"])
    ]

    colors = ["#d62728" if v >= 0 else "#1f77b4" for v in vals]

    ax.barh(y, vals, color=colors, edgecolor="white", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Points")
    ax.grid(axis="x", linestyle="--", alpha=0.25)
    ax.set_axisbelow(True)

    if len(vals) > 0:
        lim = np.max(np.abs(vals)) * 1.25
        offset = np.max(np.abs(vals)) * 0.04
    else:
        lim = 1
        offset = 0.1

    ax.set_xlim(-lim, lim)

    for i, v in enumerate(vals):
        if v >= 0:
            ax.text(v + offset, i, f"{v:.2f}", va="center", ha="left", fontsize=8)
        else:
            ax.text(v - offset, i, f"{v:.2f}", va="center", ha="right", fontsize=8)


def plot_example_cases(scorecard_example_df,
                       scorecard_example_detail_df,
                       save_path=None,
                       top_n=10):
    risk_name_map = {
        "low_risk_negative": "Low risk",
        "warning_negative_followup": "Warning",
        "positive_by_scorecard": "Positive"
    }

    n_cases = len(scorecard_example_df)
    fig, axes = plt.subplots(
        1,
        n_cases,
        figsize=(6 * n_cases, 4.8),
        squeeze=False
    )
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, scorecard_example_df.iterrows()):
        row_id = row["row_id"]
        risk_group = row["risk_group"]

        one_detail_df = scorecard_example_detail_df[
            scorecard_example_detail_df["row_id"] == row_id
        ].copy()

        title = (
            f"{risk_name_map[risk_group]}\n"
            f"score={row['score_raw']:.2f}   "
            f"prob={row['scorecard_prob']:.3f}\n"
            f"(cutoffs: {score_cut_warn_round}, {score_cut_positive_round})"
        )

        plot_one_example_case(
            ax=ax,
            detail_df=one_detail_df,
            title=title,
            top_n=top_n
        )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=180, bbox_inches="tight")

    plt.show()


plot_example_cases(
    scorecard_example_df=scorecard_example_df,
    scorecard_example_detail_df=scorecard_example_detail_df,
    save_path=scorecard_dir / "scorecard_example_cases.png",
    top_n=10
)

In [ ]:
for model_name, y_prob in test_pred_prob_dict.items():
    print(
        model_name,
        "mean_pred =", np.mean(y_prob),
        "event_rate =", np.mean(y_test_sel.values)
    )